**Algonauts 2025 Challenge — Complete Training & Submission Workflow**

This notebook integrates:
- Your MultimodalTRIBE_v2 model + BMORStream front-end
- Starter kit data loading and preprocessing
- Training on a **subset** of data for quick prototyping
- Per-parcel Pearson correlation validation (challenge metric)
- Submission formatting (nested dicts, .npy, .zip for Codabench)

**Steps Overview:**
1. Load precomputed PCA-reduced features (visual, audio, language)
2. Align features and fMRI responses
3. Train MultimodalTRIBE_v2 on a subset (1-2 episodes) with your functions
4. Validate and compute per-parcel correlations
5. Format and prepare submission for Codabench
6. Upload to Codabench for evaluation

v2 of Algonauts Projects with extended procedures and approaches

**Step 0: Checking environment setup and importing required libraries**

In [1]:
# Checking GPU availability and properties using PyTorch

import torch

# Check if CUDA is available
cuda_available = torch.cuda.is_available()
print(f"CUDA is available: {cuda_available}")

if cuda_available:
    # Get the number of CUDA devices
    n_cuda_devices = torch.cuda.device_count()
    print(f"Number of CUDA devices: {n_cuda_devices}")
    
    # Print information for each CUDA device
    for i in range(n_cuda_devices):
        device_props = torch.cuda.get_device_properties(i)
        print(f"\nCUDA Device {i}:")
        print(f"  Name: {device_props.name}")
        print(f"  Compute Capability: {device_props.major}.{device_props.minor}")
        print(f"  Total Memory: {device_props.total_memory / 1024**3:.2f} GB")
        
    # Get current device information
    current_device = torch.cuda.current_device()
    print(f"\nCurrent CUDA device: {current_device}")
else:
    print("No CUDA devices found. PyTorch will run on CPU only.")

CUDA is available: True
Number of CUDA devices: 1

CUDA Device 0:
  Name: NVIDIA GeForce RTX 4050 Laptop GPU
  Compute Capability: 8.9
  Total Memory: 6.00 GB

Current CUDA device: 0


In [3]:
# Checking system configuration
import sys
import subprocess
import torch

def check_nvidia_gpu():
    try:
        # Try to get GPU info using nvidia-smi
        output = subprocess.check_output(['nvidia-smi'], stderr=subprocess.STDOUT)
        return output.decode('utf-8')
    except:
        return "No NVIDIA GPU detected or nvidia-smi not found"

print("System Information:")
print("-" * 50)
print(f"Python Version: {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"CUDA Version: {torch.version.cuda}")
print("\nGPU Information:")
print("-" * 50)
print(check_nvidia_gpu())

System Information:
--------------------------------------------------
Python Version: 3.11.0
PyTorch Version: 2.7.1+cu118
CUDA Available: True
CUDA Version: 11.8

GPU Information:
--------------------------------------------------
Wed Nov 26 17:24:58 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.29                 Driver Version: 581.29         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |


In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

x = torch.rand(3, 3).to(device)  # tensor on GPU
print(x.device)


cuda:0


In [4]:
# Importing the required libraries (the most fun part of all the code)

import os
import json
import math
import shutil
import time
from pathlib import Path
import glob
import re
import numpy as np
import pandas as pd
import h5py
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import librosa
import ast
import string
import zipfile
from tqdm.notebook import tqdm
from sklearn.linear_model import RidgeCV, Ridge
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.stats import pearsonr
import cv2
import nibabel as nib
from nilearn import plotting
from nilearn.maskers import NiftiLabelsMasker
import ipywidgets as widgets
from ipywidgets import VBox, Dropdown, Button
from IPython.display import Video, display, clear_output
from moviepy.editor import VideoFileClip
from transformers import BertTokenizer, BertModel
from torchvision.transforms import Compose, Lambda, CenterCrop
from torchvision.models.feature_extraction import create_feature_extractor
from omegaconf import DictConfig, OmegaConf

Step 1: Understanding Challenge Data (Based on starter notebook)

The challenge data comes from the CNeuroMod dataset, and consists of multimodal movie stimuli and corresponding whole-brain time series fMRI responses of four subjects. Challenge participants will train and evaluate their encoding models using a subset of CNeuroMod's data which includes almost 80 hours of multimodal movie stimuli and corresponding fMRI responses. The stimuli consist of movie visual frames, audio samples, and time-stamped language transcripts. The neural data consist of whole-brain fMRI responses for four CNeuroMod subjects (sub-01, sub-02, sub-03 and sub-05), normalized to the Montreal Neurological Institute (MNI) spatial template (Brett et al., 2002), and processed as time series whose signal is assigned to 1,000 functionally defined brain parcels (Schaefer et al., 2018).

Multimodal movie stimuli
The multimodal (audio, visual and language) stimuli of the Algonauts 2025 challenge consist of .mkv files of audiovisual movies, and of .tsv files that contain corresponding timestamped movie transcripts.

.mkv files (audiovisual movie stimuli)
The .mkv files consist of movies that combine the visual and audio modalities, for seasons 1 to 7 of Friends and for Movie10.

Friends (seasons 1-7)
The .mkv files for seasons 1 to 7 of Friends are found at ../algonauts_2025.competitors/stimuli/movies/friends/s<season>/, and have the naming convention friends_s-<season>e<episode><episode_split>.mkv, where:

season: Number indicating the Friends season.
episode: Number indicating the Friends episode.
episode_split: Full episodes were split into shorter (~12 min) segments watched by participants inside the MRI in order to reduce the duration of fMRI data acquisition runs. Letters indicate the split of each episode. Most Friends episodes are split into two parts (i.e., splits a and b), but a handful of longer episodes are split into four parts (i.e., splits a, b, c and d).
Movie10
The .mkv files for Movie10 are found at ../algonauts_2025.competitors/stimuli/movies/movie10/<movie>/, and have the naming convention <movie><movie_split>.mkv, where:

movie: String indicating the movie name.
movie_split: Number indicating the movie split. Each movie was split into several segments to limit the duration of consecutive fMRI data acquisition runs.

.tsv files (timestamped movie transcripts)
The .tsv files contain the timestamped movie transcripts, that is, transcripts of spoken content (dialogue) in the movie stimuli, for seasons 1 to 7 of Friends and for Movie10.

Friends (seasons 1-7)
The .tsv files for seasons 1 to 7 of Friends are found at ../algonauts_neuromod.competitors/stimuli/transcripts/friends/s<season>/, and have the naming convention friends_s-<season>e<episode><episode_split>.tsv, where:

season: Number indicating the Friends season.
episode: Number indicating the Friends episode.
episode_split: Letter indicating the split of the episode. Most Friends episodes are split into two parts (i.e., splits a and b), but a handful of longer episodes are split into four parts (i.e., splits a, b, c and d).
Movie10
The .tsv files for Movie10 are found at ../algonauts_neuromod.competitors/stimuli/transcripts/movie10/<movie>/, and have the naming convention movie10_<movie><movie_split>.tsv, where:

movie: String indicating the movie name.
movie_split: Number indicating the movie split.
.tsv file content
The .tsv files splits transcribed movie dialogue into chunks of 1.49 seconds, where each row of the .tsv file corresponds to one such chunk. This segmentation was performed to facilitate alignment with the fMRI data, since fMRI volumes were acquired with a repetition time (TR) of 1.49 seconds (that is, one fMRI sample was acquired every 1.49 seconds). If no words were spoken during a specific chunk, the corresponding .tsv file row will be empty.

The columns of the .tsv files consist of different attributes of the language transcripts:

text_per_tr: Sentence consisting of words that were spoken during the chunk of interest (i.e., words with word offset within the chunk of interest, even if their onset was in the previous chunk).
words_per_tr: List of individual words that were spoken during the chunk of interest.
onsets_per_tr: Starting time (in seconds) of each word spoken during the chunk, relative to movie onset.
durations_per_tr: Duration (in seconds) of each word spoken during the chunk.
NOTE: Since the transcribed movie dialogue is not split based on word onset/offset (but rather into chunks of 1.49 seconds), the onset and offset of some words might fall in different (consecutive) chunks. In case a word is split into two consecutive chunks, it will be assigned to the chunk of word offset.

**Step 2: Data Pre-Processing and Loading**

In [ ]:
# Data Loading functions to fetch .mkv and .tsv files



In [5]:
# Functions to neccessiate alignment of .mkv movies with the .tsv transcripts

def load_transcript(transcript_path):
    """
    Loads a transcript file and returns it as a DataFrame.

    Parameters
    ----------
    transcript_path : str
        Path to the .tsv transcript file.

    """
    df = pd.read_csv(transcript_path, sep='\t')
    return df


def get_movie_info(movie_path):
    """
    Extracts the frame rate (FPS) and total duration of a movie.

    Parameters
    ----------
    movie_path : str
        Path to the .mkv movie file.

    """

    cap = cv2.VideoCapture(movie_path)
    fps, frame_count = cap.get(cv2.CAP_PROP_FPS), cap.get(cv2.CAP_PROP_FRAME_COUNT)
    cap.release()

    return fps, frame_count / fps


def split_movie_into_chunks(movie_path, chunk_duration=1.49):
    """
    Divides a video into fixed-duration chunks.

    Parameters
    ----------
    movie_path : str
        Path to the .mkv movie file.
    chunk_duration : float, optional
        Duration of each chunk in seconds (default is 1.49).

    """

    _, video_duration = get_movie_info(movie_path)
    chunks = []
    start_time = 0.0

    # Create chunks for the specified time
    while start_time < video_duration:
        end_time = min(start_time + chunk_duration, video_duration)
        chunks.append((start_time, end_time))
        start_time += chunk_duration
    return chunks

def extract_movie_segment_with_sound(movie_path, start_time, end_time,
    output_path='output_segment.mp4'):
    """
    Extracts a specific segment of a video with sound and saves it.

    Parameters
    ----------
    movie_path : str
        Path to the .mkv movie file.
    start_time : float
        Start time of the segment in seconds.
    end_time : float
        End time of the segment in seconds.
    output_path : str, optional
        Path to save the output segment (default is 'output_segment.mp4').

    """

    # Create movie segment
    movie_segment = VideoFileClip(movie_path).subclip(start_time, end_time)
    print(f"\nWriting movie file from {start_time}s until {end_time}s")

    # Write video file
    movie_segment.write_videofile(output_path, codec="libx264",
        audio_codec="aac", verbose=False, logger=None)
    return output_path


def display_transcript_and_movie(chunk_index, transcript_df, chunks,
    movie_path):
    """
    Displays transcript, movie, onset, and duration for a selected chunk.

    Parameters
    ----------
    chunk_index : int
        Index of the selected chunk.
    transcript_df : DataFrame
        DataFrame containing transcript data.
    chunks : list
        List of (start_time, end_time) tuples for video chunks.
    movie_path : str
        Path to the .mkv movie file.

    """
    # Retrieve the start and end times for the selected chunk
    start_time, end_time = chunks[chunk_index]

    # Get the corresponding transcript row if it exists in the DataFrame
    transcript_chunk = transcript_df.iloc[chunk_index] if chunk_index < len(transcript_df) else None

    # Display the stimulus chunk number
    print(f"\nChunk number: {chunk_index + 1}")

    # Display transcript details if available; otherwise, indicate no dialogue
    if transcript_chunk is not None and pd.notna(transcript_chunk['text_per_tr']):
        print(f"\nText: {transcript_chunk['text_per_tr']}")
        print(f"Words: {transcript_chunk['words_per_tr']}")
        print(f"Onsets: {transcript_chunk.get('onsets_per_tr', 'N/A')}")
        print(f"Durations: {transcript_chunk.get('durations_per_tr', 'N/A')}")
    else:
        print("<No dialogue in this scene>")

    # Extract and display the video segment
    output_movie_path = extract_movie_segment_with_sound(movie_path, start_time,
        end_time)
    display(Video(output_movie_path, embed=True, width=640, height=480))


def create_dropdown_by_text(transcript_df):
    """
    Creates a dropdown widget for selecting chunks by their text.

    Parameters
    ----------
    transcript_df : DataFrame
        DataFrame containing transcript data.

    """

    options = []

    # Iterate over each row in the transcript DataFrame
    for i, row in transcript_df.iterrows():
        if pd.notna(row['text_per_tr']):  # Check if the transcript text is not NaN
            options.append((row['text_per_tr'], i))
        else:
            options.append(("<No dialogue in this scene>", i))
    return widgets.Dropdown(options=options, description='Select scene:')


def interface_display_transcript_and_movie(movie_path, transcript_path):
    """
    Interactive interface to align movie and transcript chunks.

    Parameters
    ----------
    movie_path : str
        Path to the .mkv movie file.
    transcript_path : str
        Path to the transcript file (.tsv).

    """

    # Load the transcript data from the provided path
    transcript_df = load_transcript(transcript_path)

    # Split the video file into chunks of 1.49 seconds
    chunks = split_movie_into_chunks(movie_path)

    # Create a dropdown widget with transcript text as options
    dropdown = create_dropdown_by_text(transcript_df)

    # Create an output widget to display video and transcript details
    output = widgets.Output()

    # Display the dropdown and output widgets
    display(dropdown, output)

    # Define the function to handle dropdown value changes
    def on_chunk_select(change):
        with output:
            output.clear_output()  # Clears previous content
            chunk_index = dropdown.value
            display_transcript_and_movie(chunk_index, transcript_df, chunks,
                movie_path)

    dropdown.observe(on_chunk_select, names='value')

In [ ]:
# HRF delay parameter
hrf_delay = 3  #@param {type:"slider", min:0, max:10, step:1}

root_data_dir = r"C:\Projects\fmri-algonauts-2025\fmri-algonauts-2025 data"

# Define file paths and dataset name
movie_path = root_data_dir + "/algonauts_2025.competitors/stimuli/movies/friends/s1/friends_s01e01a.mkv"
transcript_path = root_data_dir + "/algonauts_2025.competitors/stimuli/transcripts/friends/s1/friends_s01e01a.tsv"
fmri_file_path = root_data_dir + "/algonauts_2025.competitors/fmri/sub-01/func/sub-01_task-friends_space-MNI152NLin2009cAsym_atlas-Schaefer18_parcel-1000Par7Net_desc-s123456_bold.h5"
atlas_path = root_data_dir + "/algonauts_2025.competitors/fmri/sub-01/atlas/sub-01_space-MNI152NLin2009cAsym_atlas-Schaefer18_parcel-1000Par7Net_desc-dseg_parcellation.nii.gz"
dataset_name = "ses-003_task-s01e01a"


Running the experiment for 5 episodes instead of 1 to test my model

In [ ]:
# Create a list of file specifications
files_to_process = [
    {
        "episode": "s01e01a",
        "season": "s1",
        "subject": "sub-01",
        "hrf_delay": 3,
    },
    {
        "episode": "s01e01b",
        "season": "s1",
        "subject": "sub-02",
        "hrf_delay": 3,
    },
    {
        "episode": "s01e01b",
        "season": "s1",
        "subject": "sub-02",
        "hrf_delay": 3,
    },
    {
        "episode": "s01e01b",
        "season": "s1",
        "subject": "sub-02",
        "hrf_delay": 3,
    },
    {
        "episode": "s01e01b",
        "season": "s1",
        "subject": "sub-02",
        "hrf_delay": 3,
    }
]

# Create a dropdown to select which file to process
from ipywidgets import Dropdown

file_selector = Dropdown(
    options=[f['episode'] for f in files_to_process],
    description='Select File:',
)
display(file_selector)

In [7]:
# Align the .mkv movies and .tsv language transcripts
interface_display_transcript_and_movie(movie_path, transcript_path)

Dropdown(description='Select scene:', options=(('<No dialogue in this scene>', 0), ('<No dialogue in this scen…

Output()

In [8]:
# Brain visualization functions with fmri data mapping to brain regions

def plot_fmri_on_brain(chunk_index, fmri_file_path, atlas_path, dataset_name,
    hrf_delay):
    """
    Map fMRI responses to brain parcels and plot it on a glass brain.

    Parameters
    ----------
    chunk_index : pandas.Series
        The selected chunk from the transcript, used to determine the fMRI
        sample.
    fmri_file_path : str
        Path to the HDF5 file containing fMRI data.
    atlas_path : str
        Path to the atlas NIfTI file.
    dataset_name : str
        Name of the dataset inside the HDF5 file.
    hrf_delay : int
        fMRI detects the BOLD (Blood Oxygen Level Dependent) response, a signal
        that reflects changes in blood oxygenation levels in response to
        activity in the brain. Blood flow increases to a given brain region in
        response to its activity. This vascular response, which follows the
        hemodynamic response function (HRF), takes time. Typically, the HRF
        peaks around 5–6 seconds after a neural event: this delay reflects the
        time needed for blood oxygenation changes to propagate and for the fMRI
        signal to capture them. Therefore, this parameter introduces a delay
        between stimulus chunks and fMRI samples for a better correspondence
        between input stimuli and the brain response. For example, with a
        hrf_delay of 3, if the stimulus chunk of interest is 17, the
        corresponding fMRI sample will be 20.

    """

    print(f"\nLoading fMRI file: {fmri_file_path}")

    # Load the atlas image
    atlas_img = nib.load(atlas_path)
    atlas_data = atlas_img.get_fdata()

    # Open the fMRI reeponses file, and extract the specific dataset
    with h5py.File(fmri_file_path, 'r') as f:
        print(f"Opening fMRI dataset: {dataset_name}")
        fmri_data = f[dataset_name][()]
        print(f"fMRI dataset shape: {fmri_data.shape}")

    # Extract the corresponding sample from the fMRI responses based on the
    # selected transcript chunk, and on the hrf_delay
    if (chunk_index + hrf_delay) > len(fmri_data):
        selected_sample = len(fmri_data)
    else:
        selected_sample = chunk_index + hrf_delay
    fmri_sample_data = fmri_data[selected_sample]
    print(f"Extracting fMRI sample {selected_sample+1}.")

    # Map fMRI sample values to the brain parcels in the atlas
    output_data = np.zeros_like(atlas_data)
    for parcel_index in range(1000):
        output_data[atlas_data == (parcel_index + 1)] = \
            fmri_sample_data[parcel_index]

    # Create the output NIfTI image
    output_img = nib.Nifti1Image(output_data, affine=atlas_img.affine)

    # Plot the glass brain with the mapped fMRI data
    display = plotting.plot_glass_brain(
        output_img,
        display_mode='lyrz',
        cmap='inferno',
        colorbar=True,
        plot_abs=False)
    colorbar = display._cbar
    colorbar.set_label("fMRI activity", rotation=90, labelpad=12, fontsize=12)
    plotting.show()

In [9]:
# Main interactive interface with brain visualization
def interface_display_transcript_movie_brain(movie_path, transcript_path,
    fmri_file_path, atlas_path, dataset_name, hrf_delay):
    """
    Interactive interface to display movie and transcripts chunks along with
    the fMRI response from the corresponding sample.

    This code uses functions from Section 1.2.3.

    Parameters
    ----------
    movie_path : str
        Path to the .mkv movie file.
    transcript_path : str
        Path to the .tsv transcript file.
    fmri_file_path : str
        Path to the fMRI data file.
    atlas_path : str
        Path to the brain atlas file.
    dataset_name : str
        Name of the dataset to display fMRI data from.
    hrf_delay : int
        fMRI detects the BOLD (Blood Oxygen Level Dependent) response, a signal
        that reflects changes in blood oxygenation levels in response to
        activity in the brain. Blood flow increases to a given brain region in
        response its activity. This vascular response, which follows the
        hemodynamic response function (HRF), takes time. Typically, the HRF
        peaks around 5–6 seconds after a neural event: this delay reflects the
        time needed for blood oxygenation changes to propagate and for the fMRI
        signal to capture them. Therefore, this parameter introduces a delay
        between stimulus chunks and fMRI samples. For example, with a hrf_delay
        of 3, if the stimulus chunk of interest is 17, the corresponding fMRI
        sample will be 20.

    """

    # Load the .tsv transcript data from the provided path
    transcript_df = load_transcript(transcript_path)  # from 1.2.3

    # Split the .mkv movie file into chunks of 1.49 seconds
    chunks = split_movie_into_chunks(movie_path)  # from 1.2.3

    # Create a dropdown widget with transcript text as options
    dropdown = create_dropdown_by_text(transcript_df)  # from 1.2.3

    # Create an output widget to display video, transcript, and brain
    # visualization
    output = widgets.Output()

    # Define the function to handle dropdown value changes
    def on_chunk_select(change):
        with output:
            output.clear_output()  # Clear the previous output
            chunk_index = dropdown.value

            # Display video chunk and transcript
            display_transcript_and_movie(chunk_index, transcript_df, chunks,
                movie_path)  # from 1.2.3

            # Visualize brain fMRI data
            plot_fmri_on_brain(chunk_index, fmri_file_path, atlas_path,
                dataset_name, hrf_delay)

    dropdown.observe(on_chunk_select, names='value')
    display(dropdown, output)

In [10]:

# Get the selected transcript row/chunk from the interface
interface_display_transcript_movie_brain(movie_path, transcript_path,
    fmri_file_path, atlas_path, dataset_name, hrf_delay)

Dropdown(description='Select scene:', options=(('<No dialogue in this scene>', 0), ('<No dialogue in this scen…

Output()

**Step 3: Feature Extration of video, audio and text features**

*Video Feature Extraction*

In [12]:
def get_vision_model(device):
    """
    Load a pre-trained slow_r50 video model and set up the feature extractor.

    Parameters
    ----------
    device : torch.device
        The device on which the model will run (i.e., 'cpu' or 'cuda').

    Returns
    -------
    feature_extractor : torch.nn.Module
        The feature extractor model.
    model_layer : str
        The layer from which visual features will be extracted.

    """

    # Load the model
    model = torch.hub.load('facebookresearch/pytorchvideo', 'slow_r50',
        pretrained=True)

    # Select 'blocks.5.pool' as the feature extractor layer
    model_layer = 'blocks.5.pool'
    feature_extractor = create_feature_extractor(model,
        return_nodes=[model_layer])
    feature_extractor.to(device)
    feature_extractor.eval()

    return feature_extractor, model_layer

feature_extractor, model_layer = get_vision_model(device)

Using cache found in C:\Users\Pratik/.cache\torch\hub\facebookresearch_pytorchvideo_main


In [ ]:
def extract_visual_features(episode_path, tr, feature_extractor, model_layer,
    transform, device, save_dir_temp, save_dir_features):
    """
    Extract visual features from a movie using a pre-trained video model.

    Parameters
    ----------
    episode_path : str
        Path to the movie file for which the visual features are extracted.
    tr : float
        Duration of each chunk, in seconds (aligned with the fMRI repetition
        time, or TR).
    feature_extractor : torch.nn.Module
        Pre-trained feature extractor model.
    model_layer : str
        The model layer from which the visual features are extracted.
    transform : torchvision.transforms.Compose
        Transformation pipeline for processing video frames.
    device : torch.device
        Device for computation ('cpu' or 'cuda').
    save_dir_temp : str
        Directory where the chunked movie clips are temporarily stored for
        feature extraction.
    save_dir_features : str
        Directory where the extracted visual features are saved.

    Returns
    -------
    visual_features : float
        Array containing the extracted visual features.

    """

    # Get the onset time of each movie chunk
    clip = VideoFileClip(episode_path)
    start_times = [x for x in np.arange(0, clip.duration, tr)][:-1]
    # Create the directory where the movie chunks are temporarily saved
    temp_dir = os.path.join(save_dir_temp, 'temp')
    os.makedirs(temp_dir, exist_ok=True)

    # Empty features list
    visual_features = []

    # Loop over chunks
    with tqdm(total=len(start_times), desc="Extracting visual features") as pbar:
        for start in start_times:

            # Divide the movie in chunks of length TR, and save the resulting
            # clips as '.mp4' files
            clip_chunk = clip.subclip(start, start+tr)
            chunk_path = os.path.join(temp_dir, 'visual_chunk.mp4')
            clip_chunk.write_videofile(chunk_path, verbose=False, audio=False,
                logger=None)
            # Load the frames from the chunked movie clip
            video_clip = VideoFileClip(chunk_path)
            chunk_frames = [frame for frame in video_clip.iter_frames()]

            # Format the frames to shape:
            # (batch_size, channels, num_frames, height, width)
            frames_array = np.transpose(np.array(chunk_frames), (3, 0, 1, 2))
            # Convert the video frames to tensor
            inputs = torch.from_numpy(frames_array).float()
            # Preprocess the video frames
            inputs = transform(inputs).unsqueeze(0).to(device)

            # Extract the visual features
            with torch.no_grad():
                preds = feature_extractor(inputs)
            visual_features.append(np.reshape(preds[model_layer].cpu().numpy(), -1))

            # Update the progress bar
            pbar.update(1)

    # Convert the visual features to float32
    visual_features = np.array(visual_features, dtype='float32')

    # Save the visual features
    #out_file_visual = os.path.join(
    #    save_dir_features, f'friends_s01e01a_features_visual.h5')
    #with h5py.File(out_file_visual, 'a' if Path(out_file_visual).exists() else 'w') as f:
    #    group = f.create_group("s01e01a")
    #    group.create_dataset('visual', data=visual_features, dtype=np.float32)
    #print(f"Visual features saved to {out_file_visual}")

    # Output
    return visual_features

In [ ]:
# As an exemple, extract visual features for season 1, episode 1 of Friends
episode_path = root_data_dir + "/algonauts_2025.competitors/stimuli/movies/friends/s1/friends_s01e01a.mkv"

# Duration of each movie chunk, aligned with the fMRI TR of 1.49 seconds
tr = 1.49

# Saving directories
save_dir_temp = "./visual_features"
save_dir_features = root_data_dir +  "/stimulus_features/raw/visual/"

# Execute visual feature extraction
visual_features = extract_visual_features(episode_path, tr, feature_extractor,
    model_layer, transform, device, save_dir_temp, save_dir_features)

In [ ]:
# Print the features shape
print("Visual features shape for 'friends_s01e01a.mkv':")
print(visual_features.shape)
print('(Movie samples × Visual features length)')

# Visualize the features for five movie chunks
print("\nVisual feature vectors for 5 movie chunks:\n")
print(visual_features[20:25])

*Audio Feature Extraction*

In [ ]:
def extract_audio_features(episode_path, tr, sr, device, save_dir_temp,
    save_dir_features):
    """
    Extract audio features from a movie using Mel-frequency cepstral
    coefficients (MFCCs).

    Parameters
    ----------
    episode_path : str
        Path to the movie file for which the audio features are extracted.
    tr : float
        Duration of each chunk, in seconds (aligned with the fMRI repetition
        time, or TR).
    sr : int
        Audio sampling rate.
    device : str
        Device to perform computations ('cpu' or 'gpu').
    save_dir_temp : str
        Directory where the chunked movie clips are temporarily stored for
        feature extraction.
    save_dir_features : str
        Directory where the extracted audio features are saved.

    Returns
    -------
    audio_features : float
        Array containing the extracted audio features.

    """

    # Get the onset time of each movie chunk
    clip = VideoFileClip(episode_path)
    start_times = [x for x in np.arange(0, clip.duration, tr)][:-1]
    # Create the directory where the movie chunks are temporarily saved
    temp_dir = os.path.join(save_dir_temp, 'temp')
    os.makedirs(temp_dir, exist_ok=True)

    # Empty features list
    audio_features = []

    ### Loop over chunks ###
    with tqdm(total=len(start_times), desc="Extracting audio features") as pbar:
        for start in start_times:

            # Divide the movie in chunks of length TR, and save the resulting
            # audio clips as '.wav' files
            clip_chunk = clip.subclip(start, start+tr)
            chunk_audio_path = os.path.join(temp_dir, 'audio_s01e01a.wav')
            clip_chunk.audio.write_audiofile(chunk_audio_path, verbose=False,
                logger=None)
            # Load the audio samples from the chunked movie clip
            y, sr = librosa.load(chunk_audio_path, sr=sr, mono=True)

            # Extract the audio features (MFCC)
            mfcc_features = np.mean(librosa.feature.mfcc(y=y, sr=sr), axis=1)
            audio_features.append(mfcc_features)
            # Update the progress bar
            pbar.update(1)

    ### Convert the visual features to float32 ###
    audio_features = np.array(audio_features, dtype='float32')

    # Save the audio features
    #out_file_audio = os.path.join(
    #    save_dir_features, f'friends_s01e01a_features_audio.h5')
    #with h5py.File(out_file_audio, 'a' if Path(out_file_audio).exists() else 'w') as f:
    #    group = f.create_group("s01e01a")
    #    group.create_dataset('audio', data=audio_features, dtype=np.float32)
    #print(f"Audio features saved to {out_file_audio}")

    ### Output ###
    return audio_features

In [ ]:
# As an example, extract audio features using season 1, episode 1 of Friends
episode_path = root_data_dir + "/algonauts_2025.competitors/stimuli/movies/friends/s1/friends_s01e01a.mkv"

# Duration of each movie chunk, aligned with the fMRI TR of 1.49 seconds
tr = 1.49

# Audio sampling rate
sr = 22050

# Saving directories
save_dir_temp = "./audio_features"
save_dir_features = root_data_dir +  "/stimulus_features/raw/audio/"

# Execute audio feature extraction
audio_features = extract_audio_features(episode_path, tr, sr, device,
    save_dir_temp, save_dir_features)

In [ ]:
# Print the features shape
print("Audio features shape for 'friends_s01e01a.mkv':")
print(audio_features.shape)
print('(Movie samples × Audio features length)')

# Visualize the features for five movie chunks
print("\nAudio feature vectors for 5 movie chunks:\n")
print(audio_features[20:25])

*Text Feature Extraction*

In [ ]:
def get_language_model(device):
    """
    Load a pre-trained bert-base-uncased language model and its corresponding
    tokenizer.

    Parameters
    ----------
    device : torch.device
        Device on which the model will run (e.g., 'cpu' or 'cuda').

    Returns
    -------
    model : object
        Pre-trained language model.
    tokenizer : object
        Tokenizer corresponding to the language model.

    """

    ### Load the model ###
    model = BertModel.from_pretrained('bert-base-uncased')
    model.eval().to(device)

    ### Load the tokenizer ###
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased',
        do_lower_case=True)

    ### Output ###
    return model, tokenizer

# Load the model and tokenizer
model, tokenizer = get_language_model(device)

In [ ]:
def extract_language_features(episode_path, model, tokenizer, num_used_tokens,
    kept_tokens_last_hidden_state, device, save_dir_features):
    """
    Extract language features from a movie using a pre-trained language model.

    Parameters
    ----------
    episode_path : str
        Path to the movie transcripts for which the language features are
        extracted.
    model : object
        Pre-trained language model.
    tokenizer : object
        Tokenizer corresponding to the language model.
    num_used_tokens : int
        Total number of tokens that are fed to the language model for each
        chunk, including the tokens from the chunk of interest plus N tokens
        from previous chunks (the maximum allowed by the model is 510).
    kept_tokens_last_hidden_state : int
        Number of features retained for the last_hidden_state, where each
        feature corresponds to a token, starting from the most recent token.
    device : str
        Device to perform computations ('cpu' or 'gpu').
    save_dir_features : str
        Directory where the extracted language features are saved.

    Returns
    -------
    pooler_output : list
        List containing the pooler_output features for each chunk.
    last_hidden_state : list
        List containing the last_hidden_state features for each chunk

    """

    ### Load the transcript ###
    df = pd.read_csv(episode_path, sep='\t')
    df.insert(loc=0, column="is_na", value=df["text_per_tr"].isna())

    ### Initialize the tokens and features lists ###
    tokens, np_tokens, pooler_output, last_hidden_state = [], [], [], []

    ### Loop over text chunks ###
    for i in tqdm(range(df.shape[0]), desc="Extracting language features"):

        ### Tokenize raw text ###
        if not df.iloc[i]["is_na"]: # Only tokenize if words were spoken during a chunk (i.e., if the chunk is not empty)
            # Tokenize raw text with puntuation (for pooler_output features)
            tr_text = df.iloc[i]["text_per_tr"]
            tokens.extend(tokenizer.tokenize(tr_text))
            # Tokenize without punctuation (for last_hidden_state features)
            tr_np_tokens = tokenizer.tokenize(
                tr_text.translate(str.maketrans('', '', string.punctuation)))
            np_tokens.extend(tr_np_tokens)

        ### Extract the pooler_output features ###
        if len(tokens) > 0: # Only extract features if there are tokens available
            # Select the number of tokens used from the current and past chunks,
            # and convert them into IDs
            used_tokens = tokenizer.convert_tokens_to_ids(
                tokens[-(num_used_tokens):])
            # IDs 101 and 102 are special tokens that indicate the beginning and
            # end of an input sequence, respectively.
            input_ids = [101] + used_tokens + [102]
            tensor_tokens = torch.tensor(input_ids).unsqueeze(0).to(device)
            # Extract and store the pooler_output features
            with torch.no_grad():
                outputs = model(tensor_tokens)
                pooler_output.append(outputs['pooler_output'][0].cpu().numpy())
        else: # Store NaN values if no tokes are available
            pooler_output.append(np.full(768, np.nan, dtype='float32'))

        ### Extract the last_hidden_state features ###
        if len(np_tokens) > 0: # Only extract features if there are tokens available
            np_feat = np.full((kept_tokens_last_hidden_state, 768), np.nan, dtype='float32')
            # Select the number of tokens used from the current and past chunks,
            # and convert them into IDs
            used_tokens = tokenizer.convert_tokens_to_ids(
                np_tokens[-(num_used_tokens):])
            # IDs 101 and 102 are special tokens that indicate the beginning and
            # end of an input sequence, respectively.
            np_input_ids = [101] + used_tokens + [102]
            np_tensor_tokens = torch.tensor(np_input_ids).unsqueeze(0).to(device)
            # Extract and store the last_hidden_state features
            with torch.no_grad():
                np_outputs = model(np_tensor_tokens)
                np_outputs = np_outputs['last_hidden_state'][0][1:-1].cpu().numpy()
            tk_idx = min(kept_tokens_last_hidden_state, len(np_tokens))
            np_feat[-tk_idx:, :] = np_outputs[-tk_idx:]
            last_hidden_state.append(np_feat)
        else: # Store NaN values if no tokens are available
            last_hidden_state.append(np.full(
                (kept_tokens_last_hidden_state, 768), np.nan, dtype='float32'))

    ### Convert the language features to float32 ###
    pooler_output = np.array(pooler_output, dtype='float32')
    last_hidden_state = np.array(last_hidden_state, dtype='float32')

    ### Save the language features ###
    #out_file_language = os.path.join(
    #    save_dir_features, f'friends_s01e01a_features_language.h5')
    #with h5py.File(out_file_language, 'a' if Path(out_file_language).exists() else 'w') as f:
    #    group = f.create_group("s01e01a")
    #    group.create_dataset('language_pooler_output', data=pooler_output,
    #        dtype=np.float32)
    #    group.create_dataset('language_last_hidden_state',
    #        data=last_hidden_state, dtype=np.float32)
    #print(f"Language features saved to {out_file_language}")

    ### Output ###
    return pooler_output, last_hidden_state

In [ ]:
# As an exemple, extract language features using season 1, episode 1 of Friends
episode_path = root_data_dir + "/algonauts_2025.competitors/stimuli/transcripts/friends/s1/friends_s01e01a.tsv"

# Saving directory
save_dir_features = root_data_dir +  "/stimulus_features/raw/language/"

# Other parameters
num_used_tokens = 510
kept_tokens_last_hidden_state = 10

# Execute language feature extraction
pooler_output, last_hidden_state = extract_language_features(episode_path,
    model, tokenizer, num_used_tokens, kept_tokens_last_hidden_state, device,
    save_dir_features)

In [ ]:
# Print the features shape
# pooler_output
print("pooler_output features shape for 'friends_s01e01a.mkv':")
print(pooler_output.shape)
print('(Movie samples × pooler_output features length)')
# last_hidden_state
print("\nlast_hidden_state features shape for 'friends_s01e01a.mkv':")
print(last_hidden_state.shape)
print('(Movie samples × Kept tokens × pooler_output features length)')

# Visualize the features for five movie chunks
# pooler_output
print("\npooler_output features for 5 movie chunks:\n")
print(pooler_output[20:25])
# last_hidden_state
print("\nlast_hidden_state features for 5 movie chunks:\n")
print(last_hidden_state[20:25])

*Reduce the stimulus features dimensionality using PCA*

# STEP 1: Data Discovery & 10% Sampling Strategy

**Note:** This step identifies which episodes and subjects to use (10% sampling). 
In STEP 2, we will extract features using the extraction functions from earlier cells:
- **Visual**: slow_r50 model (from cell 25-26)
- **Audio**: MFCC features (from cell 29-30)  
- **Language**: BERT embeddings (from cell 32-33)

In [ ]:
import os
import glob
import h5py
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

# Root data directory
root_data_dir = r"C:\Projects\fmri-algonauts-2025\fmri-algonauts-2025 data"
algonauts_dir = os.path.join(root_data_dir, "algonauts_2025.competitors")

print("="*70)
print("STEP 1: DATA DISCOVERY & 10% SAMPLING")
print("="*70)

# Discover available episodes
print("\n[1] Scanning for available episodes...")
stimuli_dir = os.path.join(algonauts_dir, "stimuli")
transcript_dir = os.path.join(stimuli_dir, "transcripts", "friends")

# Find all available seasons and episodes
available_episodes = []
if os.path.exists(transcript_dir):
    for season_dir in sorted(os.listdir(transcript_dir)):
        season_path = os.path.join(transcript_dir, season_dir)
        if os.path.isdir(season_path):
            for transcript_file in sorted(os.listdir(season_path)):
                if transcript_file.endswith('.tsv'):
                    episode = transcript_file.replace('friends_', '').replace('.tsv', '')
                    available_episodes.append({
                        'episode': episode,
                        'season': season_dir,
                        'transcript_path': os.path.join(season_path, transcript_file),
                    })

print(f"✓ Found {len(available_episodes)} episodes")
print(f"  Episodes: {[e['episode'] for e in available_episodes[:5]]} ... (showing first 5)")

# Discover available subjects and their fMRI files
print("\n[2] Scanning for available subjects...")
fmri_base_dir = os.path.join(algonauts_dir, "fmri")
available_subjects = []

if os.path.exists(fmri_base_dir):
    for subject_dir in sorted(os.listdir(fmri_base_dir)):
        if subject_dir.startswith('sub-'):
            subject_path = os.path.join(fmri_base_dir, subject_dir)
            if os.path.isdir(subject_path):
                available_subjects.append({
                    'subject': subject_dir,
                    'fmri_dir': os.path.join(subject_path, 'func'),
                    'atlas_path': os.path.join(subject_path, 'atlas', 
                                               f'{subject_dir}_space-MNI152NLin2009cAsym_atlas-Schaefer18_parcel-1000Par7Net_desc-dseg_parcellation.nii.gz'),
                })

print(f"✓ Found {len(available_subjects)} subjects")
print(f"  Subjects: {[s['subject'] for s in available_subjects]}")

# Calculate 10% sampling
n_episodes = len(available_episodes)
n_subjects = len(available_subjects)
sample_size = max(1, int(np.ceil(n_episodes * 0.1)))  # 10% of episodes
n_samples_per_subject = max(1, int(np.ceil(n_subjects * 0.1)))  # 10% of subjects

print(f"\n[3] 10% Sampling Strategy:")
print(f"  Total episodes available: {n_episodes}")
print(f"  Sampling {sample_size} episode(s) for quick iteration")
print(f"  Total subjects available: {n_subjects}")
print(f"  Sampling {n_samples_per_subject} subject(s) for quick iteration")

# Select 10% samples
np.random.seed(42)
sampled_episode_indices = np.random.choice(n_episodes, size=sample_size, replace=False)
sampled_subject_indices = np.random.choice(n_subjects, size=n_samples_per_subject, replace=False)

sampled_episodes = [available_episodes[i] for i in sorted(sampled_episode_indices)]
sampled_subjects = [available_subjects[i] for i in sorted(sampled_subject_indices)]

print(f"\n[4] Selected Episodes (10% sample):")
for ep in sampled_episodes:
    print(f"  - {ep['episode']} (Season: {ep['season']})")

print(f"\n[5] Selected Subjects (10% sample):")
for subj in sampled_subjects:
    print(f"  - {subj['subject']}")

print(f"\n✓ Data discovery complete. Ready for ingestion.")


# STEP 2: Data Ingestion (Extract & Load Features + fMRI)

**This step integrates the feature extraction pipeline with data loading:**
1. **Visual Feature Extraction** (slow_r50): Processes movie frames at TR-aligned chunks → 2048-dim features
2. **Audio Feature Extraction** (MFCC): Computes mel-frequency cepstral coefficients → 20-dim features
3. **Language Feature Extraction** (BERT): Tokenizes transcripts with BERT → 768-dim embeddings
4. **fMRI Loading**: Reads HDF5 files with 1000-parcel responses

All extracted features are cached to avoid redundant computation on reruns.

In [ ]:
print("\n" + "="*70)
print("STEP 2: DATA INGESTION (Extract & Load Features + fMRI)")
print("="*70)

# First, extract features from raw movies/transcripts if not already saved
# Using the extraction functions defined earlier in the notebook

print("\n[1] Preparing feature extraction tools...")

# Visual feature extractor (already loaded in earlier cell)
# Reuse the feature_extractor and model_layer from cell 25

# Audio extraction parameters
sr = 22050  # Sample rate for audio
device_audio = device  # Use same device as visual

# Text extraction parameters - will use BERT
# (text extraction function should be defined in earlier cells)

print("  ✓ Feature extraction tools ready")

def extract_and_cache_features(episode_info, root_data_dir, tr=1.49):
    """
    Extract visual, audio, and language features for an episode.
    Caches results to avoid re-extraction.
    
    Parameters
    ----------
    episode_info : dict
        Episode info with 'episode' and 'season' keys
    root_data_dir : str
        Root data directory path
    tr : float
        TR duration (1.49 seconds)
    
    Returns
    -------
    dict
        Dictionary with 'visual', 'audio', 'language' feature arrays
    """
    algonauts_dir = os.path.join(root_data_dir, "algonauts_2025.competitors")
    episode_name = episode_info['episode']
    season = episode_info['season']
    
    # Cache directory
    cache_dir = os.path.join(root_data_dir, "feature_cache")
    os.makedirs(cache_dir, exist_ok=True)
    cache_file = os.path.join(cache_dir, f"{episode_name}_features.npz")
    
    # If cached, load and return
    if os.path.exists(cache_file):
        print(f"    Loading cached features for {episode_name}")
        cached = np.load(cache_file, allow_pickle=True)
        return {
            'visual': cached['visual'],
            'audio': cached['audio'],
            'language': cached['language'],
        }
    
    print(f"    Extracting features for {episode_name}...")
    episode_path = os.path.join(
        algonauts_dir, "stimuli", "movies", "friends", season, f"friends_{episode_name}.mkv"
    )
    
    features = {}
    
    # Extract visual features (using pre-loaded feature_extractor from earlier cell)
    try:
        print(f"      Extracting visual features...")
        visual_feats = extract_visual_features(
            episode_path, tr, feature_extractor, model_layer, 
            transform, device, "./temp_visual", cache_dir
        )
        features['visual'] = visual_feats
        print(f"      ✓ Visual: {visual_feats.shape}")
    except Exception as e:
        print(f"      ✗ Visual extraction failed: {e}")
        features['visual'] = None
    
    # Extract audio features (using function from earlier cell)
    try:
        print(f"      Extracting audio features...")
        audio_feats = extract_audio_features(
            episode_path, tr, sr, device_audio, "./temp_audio", cache_dir
        )
        features['audio'] = audio_feats
        print(f"      ✓ Audio: {audio_feats.shape}")
    except Exception as e:
        print(f"      ✗ Audio extraction failed: {e}")
        features['audio'] = None
    
    # Extract language features (using function from earlier cell)
    transcript_path = os.path.join(
        algonauts_dir, "stimuli", "transcripts", "friends", season, f"friends_{episode_name}.tsv"
    )
    try:
        print(f"      Extracting language features...")
        language_feats = extract_language_features(
            transcript_path, device
        )
        features['language'] = language_feats
        print(f"      ✓ Language: {language_feats.shape}")
    except Exception as e:
        print(f"      ✗ Language extraction failed: {e}")
        features['language'] = None
    
    # Cache the extracted features
    np.savez(
        cache_file,
        visual=features['visual'],
        audio=features['audio'],
        language=features['language']
    )
    print(f"      Cached to {cache_file}")
    
    return features

# Extract features for all sampled episodes
print(f"\n[2] Extracting features for {len(sampled_episodes)} sampled episode(s)...")
features_by_episode = {}

for ep in sampled_episodes:
    print(f"\n  {ep['episode']}:")
    ep_features = extract_and_cache_features(ep, root_data_dir, tr=1.49)
    
    if all(v is not None for v in ep_features.values()):
        features_by_episode[ep['episode']] = ep_features
    else:
        print(f"  ⚠ Skipping {ep['episode']}: missing some features")

print(f"\n✓ Feature extraction complete for {len(features_by_episode)} episode(s)")

# Load fMRI data (same as before - no extraction needed, just loading)
print(f"\n[3] Loading fMRI for {len(sampled_subjects)} sampled subject(s)...")
fmri_by_subject = {}

for subject in sampled_subjects:
    print(f"\n  Loading {subject['subject']}:")
    subject_fmri = {}
    
    for ep in sampled_episodes:
        ep_name = ep['episode']
        if ep_name not in features_by_episode:  # Skip if features unavailable
            continue
        
        fmri = load_fmri_for_subject_episode(subject, ep)
        if fmri is not None:
            subject_fmri[ep_name] = fmri
            print(f"    ✓ {ep_name}: shape {fmri.shape}")
        else:
            print(f"    ✗ {ep_name}: not found")
    
    if subject_fmri:
        fmri_by_subject[subject['subject']] = subject_fmri

print(f"\n✓ fMRI loading complete for {len(fmri_by_subject)} subject(s)")

# Summary
print(f"\n[4] Data Ingestion Summary:")
print(f"  Features extracted: {len(features_by_episode)} episodes")
print(f"    - Visual features extracted from slow_r50 model")
print(f"    - Audio features extracted using MFCC analysis")
print(f"    - Language features extracted from BERT embeddings")
print(f"  fMRI loaded: {len(fmri_by_subject)} subjects × episodes")
print(f"  Total (subject, episode) pairs: {sum(len(v) for v in fmri_by_subject.values())}")


# STEP 3: Preprocessing & Alignment (HRF Delay, Normalization, Concatenation)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

print("\n" + "="*70)
print("STEP 3: PREPROCESSING & ALIGNMENT")
print("="*70)

hrf_delay = 3  # fMRI delay in TRs to account for hemodynamic response

# Prepare aligned dataset
aligned_data = []

print(f"\n[1] Aligning features and fMRI with HRF delay={hrf_delay}...")

for subject in fmri_by_subject.keys():
    for episode in sampled_episodes:
        ep_name = episode['episode']
        
        # Skip if missing either features or fMRI
        if ep_name not in features_by_episode or ep_name not in fmri_by_subject[subject]:
            continue
        
        features = features_by_episode[ep_name]
        fmri = fmri_by_subject[subject][ep_name]
        
        print(f"\n  {subject} / {ep_name}:")
        print(f"    Original shapes:")
        print(f"      Visual: {features['visual'].shape if 'visual' in features else 'N/A'}")
        print(f"      Audio: {features['audio'].shape if 'audio' in features else 'N/A'}")
        print(f"      Language: {features['language'].shape if 'language' in features else 'N/A'}")
        print(f"      fMRI: {fmri.shape}")
        
        # Apply HRF delay to fMRI
        # Shift fMRI by hrf_delay samples and truncate features to match
        n_features = features[list(features.keys())[0]].shape[0]  # Get from first available feature
        
        # Align: fMRI sample i corresponds to feature sample (i - hrf_delay)
        # So we take fMRI[hrf_delay:] and features[:n_features-hrf_delay]
        if fmri.shape[0] > hrf_delay:
            fmri_aligned = fmri[hrf_delay:]
            n_aligned = min(fmri_aligned.shape[0], n_features - hrf_delay)
        else:
            n_aligned = max(0, n_features - hrf_delay)
        
        if n_aligned <= 0:
            print(f"    ⚠ Skipping: insufficient samples after HRF alignment")
            continue
        
        # Concatenate feature modalities (normalize each first)
        feature_list = []
        for modality in ['visual', 'audio', 'language']:
            if modality in features:
                feat = features[modality][:n_aligned]
                # Standardize modality
                scaler = StandardScaler()
                feat_scaled = scaler.fit_transform(feat)
                feature_list.append(feat_scaled)
        
        X_combined = np.concatenate(feature_list, axis=1)
        y_fmri = fmri_aligned[:n_aligned]
        
        print(f"    Aligned shapes:")
        print(f"      Combined features: {X_combined.shape}")
        print(f"      fMRI: {y_fmri.shape}")
        
        aligned_data.append({
            'subject': subject,
            'episode': ep_name,
            'X': X_combined,
            'y': y_fmri,
        })

print(f"\n✓ Aligned {len(aligned_data)} (subject, episode) pairs")

# Combine all data
print(f"\n[2] Combining all data...")
X_all = np.vstack([d['X'] for d in aligned_data])
y_all = np.vstack([d['y'] for d in aligned_data])

print(f"  Combined X shape: {X_all.shape}")
print(f"  Combined y shape: {y_all.shape}")

# Apply global PCA to reduce feature dimensionality (optional but recommended)
print(f"\n[3] Applying PCA preprocessing...")
pca_dim = 256  # Target PCA dimension
pca = PCA(n_components=min(pca_dim, X_all.shape[1]))
X_pca = pca.fit_transform(X_all)

print(f"  Original feature dim: {X_all.shape[1]}")
print(f"  PCA reduced dim: {X_pca.shape[1]}")
print(f"  Variance explained: {pca.explained_variance_ratio_.sum():.2%}")

# Standardize PCA features
scaler_global = StandardScaler()
X_final = scaler_global.fit_transform(X_pca)

print(f"  Final X shape (standardized): {X_final.shape}")
print(f"  Final y shape: {y_all.shape}")

print(f"\n✓ Preprocessing complete. Data ready for model architecture.")

# Store for next step
dataset_config = {
    'X_final': X_final,
    'y_final': y_all,
    'pca': pca,
    'scaler_global': scaler_global,
    'aligned_data': aligned_data,
    'n_samples': X_final.shape[0],
    'n_features': X_final.shape[1],
    'n_parcels': y_all.shape[1],
}

print(f"\n[4] Dataset Config:")
print(f"  Total samples: {dataset_config['n_samples']}")
print(f"  Feature dimension: {dataset_config['n_features']}")
print(f"  Output parcels: {dataset_config['n_parcels']}")


# STEP 4: Model Architecture Training (on 10% Real Dataset)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr
import matplotlib.pyplot as plt

print("\n" + "="*70)
print("STEP 4: MODEL ARCHITECTURE TRAINING (on 10% Real Dataset)")
print("="*70)

X_train_data = dataset_config['X_final']
y_train_data = dataset_config['y_final']

print(f"\n[1] Train/Val Split...")
# 80/20 split
X_train, X_val, y_train, y_val = train_test_split(
    X_train_data, y_train_data, test_size=0.2, random_state=42
)

print(f"  Train set: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"  Val set: {X_val.shape[0]} samples, {X_val.shape[1]} features")

# Option 1: Baseline Ridge Regression
print(f"\n[2] Option A: Baseline Ridge Regression with Cross-Validation...")
ridge_cv = RidgeCV(alphas=[0.001, 0.01, 0.1, 1.0, 10.0, 100.0], cv=5)
ridge_cv.fit(X_train, y_train)

print(f"  Best alpha: {ridge_cv.alpha_}")

# Evaluate Ridge
y_val_pred_ridge = ridge_cv.predict(X_val)
mse_ridge = mean_squared_error(y_val, y_val_pred_ridge)

# Compute per-parcel Pearson correlation
ridge_correlations = []
for parcel_idx in range(y_val.shape[1]):
    r, _ = pearsonr(y_val[:, parcel_idx], y_val_pred_ridge[:, parcel_idx])
    ridge_correlations.append(r)

ridge_corr_mean = np.mean(ridge_correlations)
ridge_corr_std = np.std(ridge_correlations)

print(f"  MSE: {mse_ridge:.4f}")
print(f"  Mean per-parcel Pearson correlation: {ridge_corr_mean:.4f} ± {ridge_corr_std:.4f}")

# Option 2: MultimodalTRIBE (from cell 40)
print(f"\n[3] Option B: MultimodalTRIBE Model...")

# Reshape data for transformer (add time dimension)
# Assume each sample can be treated as sequence of length 1 for now
n_train, n_features = X_train.shape
n_subjects_train = len(set([d['subject'] for d in aligned_data]))

# Simple linear model as initialization (to test pipeline)
# For full transformer, would need more complex setup
class SimpleEncoderModel(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim=512):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
        )
        self.decoder = nn.Linear(hidden_dim // 2, output_dim)
    
    def forward(self, x):
        h = self.encoder(x)
        y = self.decoder(h)
        return y

# Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleEncoderModel(
    input_dim=X_train.shape[1],
    output_dim=y_train.shape[1],
    hidden_dim=512
).to(device)

# Training
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

X_train_t = torch.from_numpy(X_train).float().to(device)
y_train_t = torch.from_numpy(y_train).float().to(device)
X_val_t = torch.from_numpy(X_val).float().to(device)
y_val_t = torch.from_numpy(y_val).float().to(device)

print(f"  Training on device: {device}")
print(f"  Model: SimpleEncoderModel({X_train.shape[1]} -> {y_train.shape[1]})")

best_val_loss = float('inf')
patience = 5
patience_counter = 0

for epoch in range(50):
    # Train
    model.train()
    optimizer.zero_grad()
    y_pred = model(X_train_t)
    loss = loss_fn(y_pred, y_train_t)
    loss.backward()
    optimizer.step()
    
    # Validate
    model.eval()
    with torch.no_grad():
        y_val_pred_t = model(X_val_t)
        val_loss = loss_fn(y_val_pred_t, y_val_t)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_state = model.state_dict().copy()
    else:
        patience_counter += 1
    
    if (epoch + 1) % 10 == 0:
        print(f"  Epoch {epoch+1:3d}: train_loss={loss.item():.4f}, val_loss={val_loss.item():.4f}")
    
    if patience_counter >= patience:
        print(f"  Early stopping at epoch {epoch+1}")
        model.load_state_dict(best_state)
        break

# Evaluate model
model.eval()
with torch.no_grad():
    y_val_pred_model = model(X_val_t).cpu().numpy()

mse_model = mean_squared_error(y_val, y_val_pred_model)

model_correlations = []
for parcel_idx in range(y_val.shape[1]):
    r, _ = pearsonr(y_val[:, parcel_idx], y_val_pred_model[:, parcel_idx])
    model_correlations.append(r)

model_corr_mean = np.mean(model_correlations)
model_corr_std = np.std(model_correlations)

print(f"  MSE: {mse_model:.4f}")
print(f"  Mean per-parcel Pearson correlation: {model_corr_mean:.4f} ± {model_corr_std:.4f}")

# Comparison
print(f"\n[4] Model Comparison (on 10% Real Dataset):")
print(f"  Ridge Regression:        corr={ridge_corr_mean:.4f}")
print(f"  SimpleEncoderModel:      corr={model_corr_mean:.4f}")
print(f"  Winner: {'Ridge' if ridge_corr_mean > model_corr_mean else 'SimpleEncoder'}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(ridge_correlations, bins=30, alpha=0.5, label='Ridge', edgecolor='black')
axes[0].hist(model_correlations, bins=30, alpha=0.5, label='SimpleEncoder', edgecolor='black')
axes[0].set_xlabel('Per-Parcel Pearson Correlation')
axes[0].set_ylabel('Count')
axes[0].set_title('Correlation Distribution')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Scatter: Ridge predictions vs true
example_parcel = 0
axes[1].scatter(y_val[:, example_parcel], y_val_pred_ridge[:, example_parcel], 
                alpha=0.5, label='Ridge', s=20)
axes[1].scatter(y_val[:, example_parcel], y_val_pred_model[:, example_parcel], 
                alpha=0.5, label='SimpleEncoder', s=20)
lim = [y_val[:, example_parcel].min(), y_val[:, example_parcel].max()]
axes[1].plot(lim, lim, 'k--', lw=2)
axes[1].set_xlabel('True fMRI')
axes[1].set_ylabel('Predicted fMRI')
axes[1].set_title(f'Predictions vs Truth (Parcel {example_parcel})')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n✓ Training complete on 10% real dataset.")
print(f"  Samples trained: {X_train.shape[0]}")
print(f"  Samples validated: {X_val.shape[0]}")


**Step 4: Model Architecture**

In [ ]:
"""
TRIBE + B-MOR End-to-end Pipeline

This single-file pipeline implements Variant A (train encoder -> extract pooled features -> B-MOR ridge)
for Algonauts-style encoding. It includes:
 - MultimodalTRIBE class (your model with `encode_only` helper)
 - training loop for encoder using a small ROI readout
 - feature extraction (writes numpy memmap)
 - B-MOR implementation using joblib Parallel
 - prediction and per-target Pearson evaluation
 - a toy-data mode so you can sanity-check the pipeline end-to-end

USAGE:
  - To run the toy demo (small synthetic data):
      python tribe_bmor_pipeline.py --mode toy

  - For real data: replace the Dataset class / loaders and run in --mode real

Dependencies: torch, numpy, scikit-learn, joblib, tqdm
"""

import os
import argparse
import math
import random
from tqdm import tqdm

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from joblib import Parallel, delayed, dump

# --------------------------
# MultimodalTRIBE (from user) with encode_only
# --------------------------
class MultimodalTRIBE(nn.Module):
    def __init__(self,
                 D_text, D_audio, D_video,
                 proj_dim=128,
                 n_subjects=5,
                 d_model=None,
                 n_parcels=50,
                 n_trs=20,
                 max_seq_len=60,
                 transformer_layers=2,
                 nheads=4,
                 ff_dim=512,
                 dropout=0.1,
                 modality_dropout_p=0.2):
        super().__init__()
        if d_model is None:
            d_model = 3 * proj_dim
        self.txt_proj = nn.Sequential(nn.Linear(D_text, proj_dim), nn.LayerNorm(proj_dim))
        self.aud_proj = nn.Sequential(nn.Linear(D_audio, proj_dim), nn.LayerNorm(proj_dim))
        self.vid_proj = nn.Sequential(nn.Linear(D_video, proj_dim), nn.LayerNorm(proj_dim))

        self.pos_emb = nn.Parameter(torch.randn(1, max_seq_len, d_model) * 0.02)
        self.subj_emb = nn.Embedding(n_subjects, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nheads, dim_feedforward=ff_dim,
            dropout=dropout, activation='gelu', batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=transformer_layers)

        self.n_trs = n_trs
        self.pool = nn.AdaptiveAvgPool1d(n_trs)
        self.readout = nn.Linear(d_model, n_parcels)
        self.subj_bias = nn.Embedding(n_subjects, n_parcels)

        self.modality_dropout_p = modality_dropout_p

    def modality_dropout(self, x_txt, x_aud, x_vid):
        if not self.training or self.modality_dropout_p <= 0.0:
            return x_txt, x_aud, x_vid
        B = x_txt.shape[0]
        mask_txt = torch.bernoulli((1 - self.modality_dropout_p) * torch.ones(B,1,1,device=x_txt.device))
        mask_aud = torch.bernoulli((1 - self.modality_dropout_p) * torch.ones(B,1,1,device=x_aud.device))
        mask_vid = torch.bernoulli((1 - self.modality_dropout_p) * torch.ones(B,1,1,device=x_vid.device))
        sum_mask = (mask_txt + mask_aud + mask_vid).squeeze()
        for i in range(B):
            if sum_mask[i] == 0:
                choice = random.choice([0,1,2])
                if choice == 0: mask_txt[i] = 1.
                elif choice == 1: mask_aud[i] = 1.
                else: mask_vid[i] = 1.
        return x_txt * mask_txt, x_aud * mask_aud, x_vid * mask_vid

    def forward(self, x_txt, x_aud, x_vid, subject_ids):
        x_txt, x_aud, x_vid = self.modality_dropout(x_txt, x_aud, x_vid)
        t_txt = self.txt_proj(x_txt)
        t_aud = self.aud_proj(x_aud)
        t_vid = self.vid_proj(x_vid)

        x = torch.cat([t_txt, t_aud, t_vid], dim=-1)
        B, fT, _ = x.shape
        pos = self.pos_emb[:, :fT, :]
        subj = self.subj_emb(subject_ids).unsqueeze(1)
        x = x + pos + subj

        x_out = self.transformer(x)
        x_perm = x_out.transpose(1,2)
        pooled = self.pool(x_perm).transpose(1,2)

        preds = self.readout(pooled)
        preds = preds + self.subj_bias(subject_ids).unsqueeze(1)
        return preds

    @torch.no_grad()
    def encode_only(self, x_txt, x_aud, x_vid, subject_ids):
        """Run forward but stop before readout. Return pooled features [B, n_trs, d_model]"""
        self.eval()
        t_txt = self.txt_proj(x_txt)
        t_aud = self.aud_proj(x_aud)
        t_vid = self.vid_proj(x_vid)
        x = torch.cat([t_txt, t_aud, t_vid], dim=-1)
        B, fT, _ = x.shape
        pos = self.pos_emb[:, :fT, :].to(x.device)
        subj = self.subj_emb(subject_ids.to(x.device)).unsqueeze(1).to(x.device)
        x = x + pos + subj
        x_out = self.transformer(x)
        x_perm = x_out.transpose(1,2)
        pooled = self.pool(x_perm).transpose(1,2)
        return pooled

# --------------------------
# Toy dataset & real dataset placeholders
# --------------------------
class ToyFMRIDataset(Dataset):
    """Synthetic dataset to sanity check the pipeline.
    Each item returns:
      x_txt: [T, D_text]
      x_aud: [T, D_audio]
      x_vid: [T, D_video]
      subject_id: scalar
      y_small: [n_trs, n_parcels_small]
      y_all: [n_trs, S_targets]
    """
    def __init__(self, N_videos=200, T=20, D_text=256, D_audio=128, D_video=512,
                 n_subjects=5, n_parcels_small=10, S_targets=100):
        self.N = N_videos
        self.T = T
        self.D_text = D_text
        self.D_audio = D_audio
        self.D_video = D_video
        self.n_subjects = n_subjects
        self.n_parcels_small = n_parcels_small
        self.S = S_targets
        rng = np.random.RandomState(42)
        # generate random modality features
        self.text = rng.randn(self.N, T, D_text).astype(np.float32)
        self.audio = rng.randn(self.N, T, D_audio).astype(np.float32)
        self.video = rng.randn(self.N, T, D_video).astype(np.float32)
        self.subj = rng.randint(0, n_subjects, size=(self.N,)).astype(np.int64)
        # synthetic ground-truth: linear readout from random weights + noise
        W = rng.randn(D_text + D_audio + D_video, self.S).astype(np.float32)
        # compute pooled features by mean over time
        pooled = np.concatenate([self.text.mean(axis=1), self.audio.mean(axis=1), self.video.mean(axis=1)], axis=1)
        Y_all = pooled.dot(W) + 0.1 * rng.randn(self.N, self.S)
        # reshape to (N, n_trs, S) repeating same value per TR for toy simplicity
        self.Y_all = np.repeat(Y_all[:, None, :], T, axis=1).astype(np.float32)
        # small ROI as first few targets
        self.Y_small = self.Y_all[:, :, :n_parcels_small]

    def __len__(self):
        return self.N

    def __getitem__(self, idx):
        return (torch.from_numpy(self.text[idx]),
                torch.from_numpy(self.audio[idx]),
                torch.from_numpy(self.video[idx]),
                torch.tensor(self.subj[idx], dtype=torch.long),
                torch.from_numpy(self.Y_small[idx]),
                torch.from_numpy(self.Y_all[idx]))

# --------------------------
# Training encoder (small ROI)
# --------------------------

def train_encoder(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader,
                  device='cuda', epochs=5, lr=1e-4, save_path='tribe_encoder_best.pth'):
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    best_val = float('inf')

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for batch in tqdm(train_loader, desc=f"Train epoch {epoch}"):
            x_txt, x_aud, x_vid, subject_ids, y_small, _ = batch
            x_txt = x_txt.to(device); x_aud = x_aud.to(device); x_vid = x_vid.to(device)
            subject_ids = subject_ids.to(device); y_small = y_small.to(device)
            optimizer.zero_grad()
            preds = model(x_txt, x_aud, x_vid, subject_ids)  # [B, T, n_parcels_small]
            loss = criterion(preds, y_small)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * x_txt.shape[0]
        train_loss /= len(train_loader.dataset)

        # validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                x_txt, x_aud, x_vid, subject_ids, y_small, _ = batch
                x_txt = x_txt.to(device); x_aud = x_aud.to(device); x_vid = x_vid.to(device)
                subject_ids = subject_ids.to(device); y_small = y_small.to(device)
                preds = model(x_txt, x_aud, x_vid, subject_ids)
                val_loss += nn.functional.mse_loss(preds, y_small, reduction='sum').item()
        val_loss /= len(val_loader.dataset)

        print(f"Epoch {epoch}: train_loss={train_loss:.6f} val_loss={val_loss:.6f}")
        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), save_path)
            print(f"Saved best encoder -> {save_path}")
    return save_path

# --------------------------
# Feature extraction (memmap)
# --------------------------
@torch.no_grad()
def extract_features_for_dataset(model: nn.Module, dataloader: DataLoader,
                                 device='cuda', out_memmap_path='X_memmap.npy'):
    model = model.to(device)
    model.eval()

    # first pass to compute total rows and d_model
    total_rows = 0
    d_model = None
    for batch in dataloader:
        x_txt, x_aud, x_vid, subject_ids, *_ = batch
        B = x_txt.shape[0]
        total_rows += B * model.n_trs
        if d_model is None:
            pooled = model.encode_only(x_txt.to(device), x_aud.to(device), x_vid.to(device), subject_ids.to(device))
            d_model = pooled.shape[-1]
    print(f"Extracting total_rows={total_rows}, d_model={d_model}")

    X_mem = np.memmap(out_memmap_path, dtype=np.float32, mode='w+', shape=(total_rows, d_model))
    write_ptr = 0
    for batch in tqdm(dataloader, desc='Extract features'):
        x_txt, x_aud, x_vid, subject_ids, *_ = batch
        B = x_txt.shape[0]
        pooled = model.encode_only(x_txt.to(device), x_aud.to(device), x_vid.to(device), subject_ids.to(device))
        pooled_flat = pooled.reshape(B * model.n_trs, d_model).cpu().numpy()
        X_mem[write_ptr:write_ptr + pooled_flat.shape[0], :] = pooled_flat
        write_ptr += pooled_flat.shape[0]
    X_mem.flush()
    print(f"Wrote features to {out_memmap_path}")
    return out_memmap_path, (total_rows, d_model)

# --------------------------
# B-MOR joblib implementation
# --------------------------

def _fit_ridge_batch(X, Y_batch, alphas, cv):
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    rc = RidgeCV(alphas=alphas, cv=cv, scoring='neg_mean_squared_error', store_cv_values=False)
    rc.fit(Xs, Y_batch)
    return {'coef': rc.coef_, 'intercept': rc.intercept_, 'alpha': rc.alpha_, 'scaler': scaler}


def fit_bmor_joblib(X, Y, n_batches=None, n_jobs=4, alphas=None, cv=5):
    if isinstance(X, str):
        X = np.memmap(X, mode='r', dtype=np.float32)
    if isinstance(Y, str):
        Y = np.memmap(Y, mode='r', dtype=np.float32)

    if alphas is None:
        alphas = np.logspace(-6, 6, 13)

    N, F = X.shape
    _, S = Y.shape
    if n_batches is None:
        n_batches = min(S, n_jobs)

    base = S // n_batches
    remainder = S % n_batches
    batches = []
    idx = 0
    for b in range(n_batches):
        size = base + (1 if b < remainder else 0)
        batches.append((idx, idx + size))
        idx += size

    def job(start, stop):
        Yb = np.asarray(Y[:, start:stop])
        Xcopy = np.asarray(X)
        return _fit_ridge_batch(Xcopy, Yb, alphas, cv)

    print(f"Starting B-MOR with {len(batches)} batches, n_jobs={n_jobs}")
    results = Parallel(n_jobs=n_jobs)(delayed(job)(s, e) for s, e in batches)

    coefs = np.vstack([r['coef'] for r in results])
    intercepts = np.concatenate([r['intercept'] for r in results])
    alphas_used = [r['alpha'] for r in results]
    return {'coefs': coefs, 'intercepts': intercepts, 'batch_results': results, 'alphas': alphas_used}

# --------------------------
# Prediction and evaluation
# --------------------------

def predict_with_bmor(X, coefs, intercepts, scaler=None):
    if scaler is not None:
        Xs = scaler.transform(X)
    else:
        Xs = X
    Y_pred = Xs.dot(coefs.T) + intercepts[None, :]
    return Y_pred


def pearson_r_per_target(Y_true, Y_pred):
    Yt = Y_true - Y_true.mean(axis=0)
    Yp = Y_pred - Y_pred.mean(axis=0)
    num = np.sum(Yt * Yp, axis=0)
    den = np.sqrt(np.sum(Yt**2, axis=0) * np.sum(Yp**2, axis=0))
    r = num / (den + 1e-12)
    return r

# --------------------------
# Helper to create memmap Y for toy dataset
# --------------------------

def create_Y_memmap_from_dataset(dataset, path):
    # dataset yields items with Y_all as last element
    total_rows = len(dataset) * dataset.T
    S = dataset.S
    Y_mem = np.memmap(path, dtype=np.float32, mode='w+', shape=(total_rows, S))
    ptr = 0
    for i in range(len(dataset)):
        *_, y_small, y_all = dataset[i]
        # y_all shape [T, S]
        arr = y_all.numpy().astype(np.float32)
        Y_mem[ptr:ptr + arr.shape[0], :] = arr
        ptr += arr.shape[0]
    Y_mem.flush()
    return path, (total_rows, S)

# --------------------------
# main orchestration
# --------------------------

def run_toy_pipeline(args):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    # toy dims
    D_text, D_audio, D_video = 64, 32, 96
    n_subjects = 4
    n_parcels_small = 8
    S_targets = 64
    n_trs = 10

    # build toy dataset
    dataset = ToyFMRIDataset(N_videos=200, T=n_trs, D_text=D_text, D_audio=D_audio, D_video=D_video,
                             n_subjects=n_subjects, n_parcels_small=n_parcels_small, S_targets=S_targets)
    # train/val split
    n_train = int(0.8 * len(dataset))
    train_ds = torch.utils.data.Subset(dataset, list(range(n_train)))
    val_ds = torch.utils.data.Subset(dataset, list(range(n_train, len(dataset))))

    batch_size = 16
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    full_train_loader = DataLoader(train_ds, batch_size=32, shuffle=False)
    full_test_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

    # model
    model = MultimodalTRIBE(D_text=D_text, D_audio=D_audio, D_video=D_video,
                            proj_dim=32, n_subjects=n_subjects, d_model=None,
                            n_parcels=n_parcels_small, n_trs=n_trs, transformer_layers=2)

    # train encoder
    save_path = train_encoder(model, train_loader, val_loader, device=device, epochs=5, lr=3e-4, save_path='tribe_toy_best.pth')

    # load & freeze
    model.load_state_dict(torch.load(save_path, map_location=device))
    for p in model.parameters():
        p.requires_grad = False
    model.eval()

    # extract features (train & test)
    X_train_path, shape_train = extract_features_for_dataset(model, full_train_loader, device=device, out_memmap_path='X_train.npy')
    X_test_path, shape_test = extract_features_for_dataset(model, full_test_loader, device=device, out_memmap_path='X_test.npy')

    # create Y memmaps
    Y_train_path, shape_Y_train = create_Y_memmap_from_dataset(torch.utils.data.Subset(dataset, list(range(n_train))), 'Y_train.npy')
    Y_test_path, shape_Y_test = create_Y_memmap_from_dataset(torch.utils.data.Subset(dataset, list(range(n_train, len(dataset)))), 'Y_test.npy')

    # load as np.memmap with proper shape
    X_train = np.memmap(X_train_path, mode='r', dtype=np.float32).reshape(shape_train)
    X_test = np.memmap(X_test_path, mode='r', dtype=np.float32).reshape(shape_test)
    Y_train = np.memmap(Y_train_path, mode='r', dtype=np.float32).reshape(shape_Y_train)
    Y_test = np.memmap(Y_test_path, mode='r', dtype=np.float32).reshape(shape_Y_test)

    # fit a global scaler & save
    scaler = StandardScaler().fit(np.asarray(X_train))
    dump(scaler, 'scaler_global.joblib')
    X_train_scaled = scaler.transform(np.asarray(X_train))

    # B-MOR
    bmor_res = fit_bmor_joblib(X_train_scaled, Y_train, n_batches=4, n_jobs=args.n_jobs, alphas=None, cv=3)
    np.save('bmor_coefs.npy', bmor_res['coefs'])
    np.save('bmor_intercepts.npy', bmor_res['intercepts'])

    # predict
    X_test_scaled = scaler.transform(np.asarray(X_test))
    Y_pred = predict_with_bmor(X_test_scaled, bmor_res['coefs'], bmor_res['intercepts'])
    r_vals = pearson_r_per_target(np.asarray(Y_test), np.asarray(Y_pred))
    print('Toy pipeline results: mean r=', np.nanmean(r_vals), 'median r=', np.nanmedian(r_vals))
    np.save('toy_r_vals.npy', r_vals)


def main():
    p = argparse.ArgumentParser()
    p.add_argument('--mode', choices=['toy', 'real'], default='toy')
    p.add_argument('--n_jobs', type=int, default=4)
    args = p.parse_args()

    if args.mode == 'toy':
        run_toy_pipeline(args)
    else:
        raise NotImplementedError('Real mode not implemented in this single-file demo. Replace Dataset and loaders.')

if __name__ == '__main__':
    main()


**Notebook Walkthrough — classes and processing (detailed)**

- **`BMORStream`**: Lightweight B‑MOR inspired front‑end per modality.
  - Input: per‑time‑step modality vector (e.g., visual frame features).
  - Internal: two small pathways (ventral/dorsal idea) fused to an output embedding.
  - Output: tensor shaped `[B, T, proj_dim]` ready for transformer fusion.

- **`HybridBMORTransformer`**: Earlier provided example that splits visual features, runs two BMOR streams, concatenates and runs a Transformer encoder for fusion. Useful as a compact baseline.

- **`MultimodalTRIBE_v2`** (new): TRIBE-style multimodal transformer with: modality dropout, optional BMOR front‑end per modality (`use_bmor=True`), positional + subject embeddings, Transformer encoder (batch_first=True), temporal pooling to `n_trs`, and subject‑specific readout.
  - Use this class as the main end‑to‑end encoder + readout. It expects inputs `(x_txt, x_aud, x_vid)` shaped `[B, T, D_modality]` and `subject_ids` shaped `[B]`.

- **PCA processing (preprocessing script)**: Reduce modality feature dimensionality (e.g., to 64–256 dims) and save as `.npy` files. This reduces GPU memory and speeds training. Use `StandardScaler` + `PCA` as in the notebook’s preprocessing cell.

- **Memory‑mapped dataset (`fMRIMemmapDataset`)**: Loads large `.npy` files with `np.memmap` and returns small slices per sample. Keeps RAM low and reads only what is needed from disk. Always call `.copy()` on memmap slices before converting to tensors.

- **Training loop**: Use `torch.cuda.amp` for mixed precision, gradient accumulation to emulate larger batches, and a validation step computing per‑parcel Pearson correlation (challenge metric).

- **Evaluation / Submission**: Save best checkpoints by validation Pearson correlation. Export predictions in required challenge format.

**Integration recommendations**:
- For limited GPU memory prefer: PCA -> memmap -> small proj_dim (64) -> use_bmor=True (if you want the biological prior) -> mixed precision -> gradient accumulation.
- For scaling to full dataset: perform offline preprocessing to `.npy` or HDF5, then use a cluster / cloud instance with larger GPU(s) or DDP.

**GPU diagnostics and practical guidance for NVIDIA RTX 4050 (6 GB)**

To inspect the GPU and get compute capability and memory info programmatically, run the code cell below. For full hardware details, `nvidia-smi` on the host is recommended.

In [2]:
# Inspect GPU device properties with PyTorch and nvidia-smi output (if available)
import subprocess
import torch

print('PyTorch CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'--- GPU {i} ---')
        print('Name:', p.name)
        print('Compute Capability:', f'{p.major}.{p.minor}')
        print('Total memory (GB):', p.total_memory/1024**3)

# Attempt to call nvidia-smi for additional info
try:
    out = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,driver_version,clocks.sm,temperature.gpu', '--format=csv,noheader,nounits'])
    print('\nnvidia-smi output:')
    print(out.decode('utf-8'))
except Exception as e:
    print('nvidia-smi not available or not found on PATH')

# Quick note: compute capability alone does not give FLOPS without SM count and clock info.
# For rough theoretical peak FLOPS you need: 2 * SMs * cores_per_SM * clock_GHz.
# Use vendor docs or `nvidia-smi -q` / device properties to get SM and clock values.

PyTorch CUDA available: True
--- GPU 0 ---
Name: NVIDIA GeForce RTX 4050 Laptop GPU
Compute Capability: 8.9
Total memory (GB): 5.99658203125

nvidia-smi output:
NVIDIA GeForce RTX 4050 Laptop GPU, 6141, 581.29, 2055, 39


nvidia-smi output:
NVIDIA GeForce RTX 4050 Laptop GPU, 6141, 581.29, 2055, 39



**Training strategy on a 6 GB GPU with 85 GB dataset**

Short checklist and best practices to get the best results given the hardware constraint:

1. Offline preprocessing (mandatory):
   - Run PCA / StandardScaler offline to reduce each modality to ~64–128 dims.
   - Save features as `float32` `.npy` files and use `np.memmap` at training time.

2. Memory‑efficient data pipeline:
   - Use `fMRIMemmapDataset` shown earlier. Keep `num_workers=0` on Windows; on Linux use 4–8 workers.

3. Model choices to fit 6 GB: 
   - Set `proj_dim=64`, `d_model=3*proj_dim`, `ff_dim=256`, `transformer_layers=2`.
   - Use `use_bmor=True` only if the extra parameters fit; otherwise start with linear projections.

4. Training tricks:
   - Use mixed precision (`torch.cuda.amp`) — reduces memory and often speeds training.
   - Use gradient accumulation (e.g., accumulate 4–8 steps) to emulate larger batch sizes.
   - Use small per‑GPU batch sizes (1–8) depending on memory.
   - Use activation checkpointing for transformer blocks via `torch.utils.checkpoint`.

5. Offload / distributed options if single GPU insufficient:
   - Use DeepSpeed ZeRO stage 2/3 with CPU offload (run on machine with more RAM).
   - Or use a cloud GPU instance (A10 / A30 / A40 / V100 / A100 families) with >=24–40 GB VRAM.

6. Data sharding and streaming:
   - Keep raw stimuli and features on disk; load minibatches via memmap.
   - Optionally prefetch next batches on CPU while GPU trains to hide IO latency.

7. Validation and checkpoints:
   - Compute Pearson per parcel and save checkpoints frequently.
   - Keep best model by validation correlation.

8. When to scale up to cloud / multi‑GPU:
   - If training time or memory prevents progress, move to a cloud instance with >=2 GPUs and use DDP.

Example local launch (single GPU):
```bash
python -u -c "from train_script import main; main()"
```
Example multi‑GPU DDP launcher (on machine with multiple GPUs or cloud):
```bash
torchrun --standalone --nproc_per_node=4 train.py --config config.yaml
```

### Data Preprocessing for Large Datasets

**IMPORTANT**: Since the full dataset (90-100GB) is too large to process in memory, a one-time preprocessing step is required. The following script should be run separately to process all your raw feature files and fMRI data. It will standardize them, reduce their dimensionality using PCA, and save them as large `.npy` files that can be loaded efficiently.

**Run this script once before proceeding (e.g., save as `preprocess_full_dataset.py` and run from your terminal):**

```python
#
# script: preprocess_full_dataset.py
#
import os
import numpy as np
import h5py
import glob
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from tqdm import tqdm

# --- CONFIGURATION ---
ROOT_DATA_DIR = r"C:\Projects\algonauts 2025 data"
OUTPUT_DIR = r"C:\Projects\algonauts 2025 data\processed_large"
N_COMPONENTS = 128
SUBJECTS = [1, 2, 3, 5] # All subjects to include

# Helper function to load features from the original HDF5 files
def load_features_from_hdf5(file_path, modality):
    with h5py.File(file_path, 'r') as data:
        for episode in data.keys(): # Assuming one episode per file
            if modality != 'language':
                features = np.asarray(data[episode][modality])
            else:
                pooler_output = np.asarray(data[episode][modality+'_pooler_output'])
                last_hidden = np.asarray(np.reshape(
                    data[episode][modality+'_last_hidden_state'], (len(pooler_output), -1)))
                features = np.append(pooler_output, last_hidden, axis=1)
    return features

def preprocess_and_reduce(all_features, n_components):
    # 1. Replace NaNs and Standardize
    features_nonan = np.nan_to_num(all_features)
    scaler = StandardScaler()
    features_std = scaler.fit_transform(features_nonan)
    
    # 2. Perform PCA
    # Ensure n_components is not greater than the number of features
    if n_components > features_std.shape[1]:
        n_components = features_std.shape[1]
    pca = PCA(n_components, random_state=42)
    features_pca = pca.fit_transform(features_std)
    return features_pca.astype('float32')

# --- MAIN SCRIPT ---
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Process stimulus features (visual, audio, language)
modalities = ['visual', 'audio', 'language']
# Adjust the glob pattern to match your file naming convention, e.g., '*features_visual.h5'
feature_files_base = os.path.join(ROOT_DATA_DIR, 'stimulus_features', 'raw')

for modality in modalities:
    print(f"--- Processing modality: {modality} ---")
    
    all_files = glob.glob(os.path.join(feature_files_base, modality, '*.h5'))
    if not all_files:
        print(f"Warning: No HDF5 files found for modality '{modality}'. Skipping.")
        continue
    
    modality_features = np.concatenate([
        load_features_from_hdf5(f, modality) for f in tqdm(all_files, desc=f"Loading {modality} files")
    ])
    
    print(f"Original full shape for {modality}: {modality_features.shape}")
    modality_pca = preprocess_and_reduce(modality_features, N_COMPONENTS)
    
    output_path = os.path.join(OUTPUT_DIR, f"{modality}_features_pca.npy")
    np.save(output_path, modality_pca)
    print(f"Saved {modality} PCA features to {output_path} with shape {modality_pca.shape}")

# 2. Process fMRI data
print("--- Processing fMRI data ---")
fmri_base_path = Path(ROOT_DATA_DIR) / 'algonauts_2025.competitors' / 'fmri'
all_fmri_data = []

fmri_files = []
for sub in SUBJECTS:
    # This glob pattern assumes your fMRI .npy files are organized by subject and session
    sub_path = fmri_base_path / f"sub-{sub:02d}" / "fmri" / "friends"
    fmri_files.extend(list(sorted(sub_path.glob("*.npy"))))

if not fmri_files:
    print("Warning: No fMRI .npy files found. Skipping fMRI processing.")
else:
    for fmri_file in tqdm(fmri_files, desc="Loading fMRI files"):
        all_fmri_data.append(np.load(fmri_file))

    full_fmri_targets = np.concatenate(all_fmri_data, axis=0).astype('float32')
    fmri_output_path = os.path.join(OUTPUT_DIR, 'fmri_targets.npy')
    np.save(fmri_output_path, full_fmri_targets)
    print(f"Saved all fMRI targets to {fmri_output_path} with shape {full_fmri_targets.shape}")

print("\nPreprocessing complete.")
```

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import os

# --- Constants and Configuration --
HRF_DELAY = 3
BATCH_SIZE = 32
PROCESSED_DATA_DIR = r"C:\Projects\algonauts 2025 data\processed_large"

# Define paths to the large preprocessed .npy files
visual_features_path = os.path.join(PROCESSED_DATA_DIR, 'visual_features_pca.npy')
audio_features_path = os.path.join(PROCESSED_DATA_DIR, 'audio_features_pca.npy')
language_features_path = os.path.join(PROCESSED_DATA_DIR, 'language_features_pca.npy')
fmri_targets_path = os.path.join(PROCESSED_DATA_DIR, 'fmri_targets.npy')


# --- Memory-Mapped Dataset Class ---
class fMRIMemmapDataset(Dataset):
    """
    PyTorch Dataset for large fMRI data using memory-mapped files.
    This dataset reads data directly from disk, keeping RAM usage low.
    """
    def __init__(self, visual_path, audio_path, text_path, fmri_path, hrf_delay):
        super().__init__()
        
        # Open the .npy files in memory-map mode
        print("Loading data with np.memmap...")
        self.visual_features = np.memmap(visual_path, dtype='float32', mode='r')
        self.audio_features = np.memmap(audio_path, dtype='float32', mode='r')
        self.text_features = np.memmap(text_path, dtype='float32', mode='r')
        self.fmri_data = np.memmap(fmri_path, dtype='float32', mode='r')
        
        self.hrf_delay = hrf_delay
        
        # Sanity checks
        assert len(self.visual_features) == len(self.audio_features) == len(self.text_features), "All feature modalities must have the same length"
        assert len(self.visual_features) == len(self.fmri_data), "Features and fMRI targets must have the same length"
        print(f"Dataset loaded. Number of samples: {len(self.visual_features)}, HRF delay: {self.hrf_delay} TRs")

    def __len__(self):
        # The total number of samples is the length of the features minus the delay,
        # as we cannot predict fMRI for the first few stimuli.
        return len(self.visual_features) - self.hrf_delay

    def __getitem__(self, idx):
        # The stimulus (features) is at index `idx`
        # The corresponding fMRI response is `hrf_delay` steps later
        target_idx = idx + self.hrf_delay

        # Read only the required slices from disk. .copy() is crucial!
        visual = self.visual_features[idx].copy()
        audio = self.audio_features[idx].copy()
        text = self.text_features[idx].copy()
        fmri = self.fmri_data[target_idx].copy()
        
        # Convert numpy arrays to PyTorch tensors
        return (torch.from_numpy(visual),
                torch.from_numpy(audio),
                torch.from_numpy(text),
                torch.from_numpy(fmri))

# --- Create Dataset and DataLoader ---

# NOTE: Make sure you have run the preprocessing script first!
try:
    dataset = fMRIMemmapDataset(
        visual_features_path,
        audio_features_path,
        language_features_path,
        fmri_targets_path,
        HRF_DELAY
    )
    
    # NOTE: For Windows, num_workers > 0 can be problematic. 
    # If you experience freezing, keep num_workers=0. On Linux, you can increase it.
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    
    print(f"\nDataLoader created successfully with batch size {BATCH_SIZE}.")
    # Example of fetching one batch to test
    vis_batch, aud_batch, txt_batch, fmr_batch = next(iter(dataloader))
    print(f"Sample visual batch shape: {vis_batch.shape}")
    print(f"Sample audio batch shape: {aud_batch.shape}")
    print(f"Sample text batch shape: {txt_batch.shape}")
    print(f"Sample fmri batch shape: {fmr_batch.shape}")

except FileNotFoundError:
    print("\nERROR: Preprocessed .npy files not found!")
    print(f"Please run the preprocessing script and ensure the following files exist in '{PROCESSED_DATA_DIR}':")
    print(f"- {os.path.basename(visual_features_path)}")
    print(f"- {os.path.basename(audio_features_path)}")
    print(f"- {os.path.basename(language_features_path)}")
    print(f"- {os.path.basename(fmri_targets_path)}")
    dataloader = None # Set to None to prevent subsequent cells from failing

**Step 5: Training**

In [ ]:
# Hyperparameters
VISUAL_DIM = 128
AUDIO_DIM = 128
TEXT_DIM = 128
HIDDEN_DIM = 512
OUTPUT_DIM = 1000
NHEAD = 8
NUM_LAYERS = 6
LEARNING_RATE = 1e-4
EPOCHS = 10
BATCH_SIZE = 32
GRADIENT_ACCUMULATION_STEPS = 2

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model, Loss, Optimizer
model = HybridBMORTransformer(VISUAL_DIM, AUDIO_DIM, TEXT_DIM, HIDDEN_DIM, OUTPUT_DIM, NHEAD, NUM_LAYERS).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
scaler = torch.cuda.amp.GradScaler()

# Training Loop
best_val_corr = -1

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    optimizer.zero_grad()
    
    for i, (visual, audio, text, fmri) in enumerate(tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}")):
        visual, audio, text, fmri = visual.to(device), audio.to(device), text.to(device), fmri.to(device)
        
        # Automatic Mixed Precision
        with torch.cuda.amp.autocast():
            outputs = model(visual, audio, text)
            loss = criterion(outputs, fmri)
            loss = loss / GRADIENT_ACCUMULATION_STEPS

        scaler.scale(loss).backward()
        
        if (i + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            
        running_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS
        
    epoch_loss = running_loss / len(dataloader)
    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {epoch_loss:.4f}")
    
    # Validation
    model.eval()
    all_preds = []
    all_fmri = []
    with torch.no_grad():
        for visual, audio, text, fmri in dataloader: # Using the same dataloader for simplicity, replace with validation loader
            visual, audio, text, fmri = visual.to(device), audio.to(device), text.to(device), fmri.to(device)
            
            with torch.cuda.amp.autocast():
                outputs = model(visual, audio, text)
            
            all_preds.append(outputs.cpu().numpy())
            all_fmri.append(fmri.cpu().numpy())
            
    all_preds = np.concatenate(all_preds)
    all_fmri = np.concatenate(all_fmri)
    
    correlations = []
    for i in range(OUTPUT_DIM):
        corr, _ = pearsonr(all_preds[:, i], all_fmri[:, i])
        if not np.isnan(corr):
            correlations.append(corr)
            
    mean_corr = np.mean(correlations)
    print(f"Validation Pearson Correlation: {mean_corr:.4f}")
    
    if mean_corr > best_val_corr:
        best_val_corr = mean_corr
        torch.save(model.state_dict(), 'best_model.pth')
        print("Saved best model")

**SECTION: Complete Training Workflow with Validation and Submission (Starter Kit Integrated)**

This section follows the Algonauts starter kit approach:
1. Train on a subset of data
2. Validate using per-parcel Pearson correlation (challenge metric)
3. Prepare submission in the required format
4. Save and zip for Codabench

In [ ]:
# ==== SUBSET TRAINING (Quick Prototyping) ====
# For fast iteration, train on just 2 episodes from Friends S01
# Then validate on Friends S06 (also 2 episodes)
# Finally prepare full Friends S7 predictions for submission

print("=" * 70)
print("SUBSET TRAINING WORKFLOW (2 episodes for fast iteration)")
print("=" * 70)

# Select a small subset for training
train_episodes_subset = ["s01e01a", "s01e01b"]
val_episodes_subset = ["s06e01a", "s06e01b"]

print(f"\nTraining episodes: {train_episodes_subset}")
print(f"Validation episodes: {val_episodes_subset}")

# For this demo, we'll use the Ridge baseline model (same as starter kit)
# But you can replace with MultimodalTRIBE_v2 by uncommenting below
USE_RIDGE_BASELINE = True  # Set to False to use your MultimodalTRIBE_v2 instead

if not USE_RIDGE_BASELINE:
    print("\nUsing MultimodalTRIBE_v2 with BMORStream front-end...")
    print("(Replace Ridge with your model training loop for production use)")


In [ ]:
# ==== STEP 1: Load subset stimulus features and fMRI responses ====
print("\n" + "="*70)
print("STEP 1: Load Stimulus Features and fMRI Responses")
print("="*70)

# Example: Load from your precomputed memmap arrays (assumed pre-processed by preprocessing script)
# OR: Use the starter kit's load_stimulus_features / load_fmri functions

# For this demo, we'll construct minimal arrays with correct shapes
# In production, load from your actual data (see commented example below)

# Example structure (replace with actual data loading):
def create_toy_subset(n_samples=100):
    """Create toy arrays for demonstration."""
    train_features = {
        'visual': np.random.randn(n_samples, 250).astype('float32'),
        'audio': np.random.randn(n_samples, 20).astype('float32'),
        'language': np.random.randn(n_samples, 250).astype('float32'),
    }
    train_fmri = np.random.randn(n_samples, 1000).astype('float32')
    
    val_features = {
        'visual': np.random.randn(n_samples, 250).astype('float32'),
        'audio': np.random.randn(n_samples, 20).astype('float32'),
        'language': np.random.randn(n_samples, 250).astype('float32'),
    }
    val_fmri = np.random.randn(n_samples, 1000).astype('float32')
    
    return train_features, train_fmri, val_features, val_fmri

# Load toy data for demonstration
train_features_subset, train_fmri_subset, val_features_subset, val_fmri_subset = create_toy_subset(100)

print(f"Train features shapes:")
for modality, feat in train_features_subset.items():
    print(f"  {modality}: {feat.shape}")
print(f"Train fMRI shape: {train_fmri_subset.shape}")

print(f"\nVal features shapes:")
for modality, feat in val_features_subset.items():
    print(f"  {modality}: {feat.shape}")
print(f"Val fMRI shape: {val_fmri_subset.shape}")

print("\nNote: Replace create_toy_subset() with actual data loading from your preprocessed .npy files")


In [ ]:
# ==== STEP 2: Train Encoding Model (Subset) ====
print("\n" + "="*70)
print("STEP 2: Train Encoding Model on Subset")
print("="*70)

# Combine all modalities for input
train_features_combined = np.concatenate([
    train_features_subset['visual'],
    train_features_subset['audio'],
    train_features_subset['language']
], axis=1)

print(f"Combined train features shape: {train_features_combined.shape}")
print(f"Train fMRI shape: {train_fmri_subset.shape}")

# Train Ridge regression model (baseline, same as starter kit)
# For production, replace with your MultimodalTRIBE_v2 training loop
from sklearn.linear_model import RidgeCV

print("\nTraining Ridge regression model...")
ridge_model = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0], cv=3)
ridge_model.fit(train_features_combined, train_fmri_subset)

print(f"Model trained successfully!")
print(f"Selected alpha: {ridge_model.alpha_}")

# ---- (Optional) Switch to MultimodalTRIBE_v2 ----
# Uncomment below to train with your custom transformer model
if False:  # Set to True to use MultimodalTRIBE_v2
    print("\nTraining MultimodalTRIBE_v2 model...")
    
    # Reshape features for transformer (add time dimension)
    B, F_total = train_features_combined.shape
    F_vis, F_aud, F_lang = 250, 20, 250
    
    txt_seq = train_features_subset['language'].reshape(B, 1, F_lang)
    aud_seq = train_features_subset['audio'].reshape(B, 1, F_aud)
    vid_seq = train_features_subset['visual'].reshape(B, 1, F_vis)
    
    # Training loop would go here
    # model = MultimodalTRIBE_v2(...)
    # ... training code using AMP, gradient accumulation, etc.


In [ ]:
# ==== STEP 3: Validate on Subset ====
print("\n" + "="*70)
print("STEP 3: Validate on Subset")
print("="*70)

# Prepare validation features
val_features_combined = np.concatenate([
    val_features_subset['visual'],
    val_features_subset['audio'],
    val_features_subset['language']
], axis=1)

print(f"Val features shape: {val_features_combined.shape}")
print(f"Val fMRI shape: {val_fmri_subset.shape}")

# Make predictions
val_predictions = ridge_model.predict(val_features_combined)
print(f"Predictions shape: {val_predictions.shape}")

# Compute per-parcel Pearson correlation
from scipy.stats import pearsonr

parcel_correlations = []
for parcel_idx in range(val_predictions.shape[1]):
    pred = val_predictions[:, parcel_idx]
    true = val_fmri_subset[:, parcel_idx]
    
    # Pearson correlation
    corr, p_value = pearsonr(pred, true)
    parcel_correlations.append(corr)

parcel_correlations = np.array(parcel_correlations)
mean_correlation = parcel_correlations.mean()
std_correlation = parcel_correlations.std()

print(f"\nValidation Results:")
print(f"  Mean per-parcel correlation: {mean_correlation:.4f}")
print(f"  Std per-parcel correlation: {std_correlation:.4f}")
print(f"  Min correlation: {parcel_correlations.min():.4f}")
print(f"  Max correlation: {parcel_correlations.max():.4f}")
print(f"  Median correlation: {np.median(parcel_correlations):.4f}")

# Plot correlation distribution
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram of correlations
axes[0].hist(parcel_correlations, bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(mean_correlation, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_correlation:.4f}')
axes[0].set_xlabel('Pearson Correlation')
axes[0].set_ylabel('Number of Parcels')
axes[0].set_title('Distribution of Per-Parcel Correlations')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Scatter: pred vs true (first parcel as example)
example_parcel = 0
axes[1].scatter(val_fmri_subset[:, example_parcel], val_predictions[:, example_parcel], alpha=0.5)
axes[1].plot([val_fmri_subset[:, example_parcel].min(), val_fmri_subset[:, example_parcel].max()],
             [val_fmri_subset[:, example_parcel].min(), val_fmri_subset[:, example_parcel].max()],
             'r--', lw=2)
axes[1].set_xlabel('True fMRI Response')
axes[1].set_ylabel('Predicted fMRI Response')
axes[1].set_title(f'Predictions vs Truth (Parcel {example_parcel})')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nValidation complete. Next: prepare submission format.")


In [ ]:
# ==== STEP 4: Prepare Submission Format (Friends S7) ====
print("\n" + "="*70)
print("STEP 4: Prepare Submission Format (Friends S7)")
print("="*70)

print("""
SUBMISSION FORMAT SPECIFICATION:
================================
The Codabench challenge requires predictions in the following format:

1. For each subject (sub-01, sub-02, sub-03, sub-04), create predictions for Friends Season 7
2. For each episode (s07e01a, s07e01b, ..., s07e12b), predict fMRI activity for all samples
3. Organize as nested dictionary: {subject: {episode: array}}
4. Each array must match the exact number of samples in the official test fMRI file
5. Save as .npy, then zip and upload to Codabench

EXAMPLE STRUCTURE:
=================
submission_dict = {
    'sub-01': {
        's07e01a': predictions_array_shape_(N_samples_s07e01a, 1000),
        's07e01b': predictions_array_shape_(N_samples_s07e01b, 1000),
        ...
    },
    'sub-02': { ... },
    ...
}
""")

# FOR DEMONSTRATION: Create toy submission using val predictions
print("\nCreating toy submission dictionary (demonstration)...")

submission_dict = {}

# This is a toy example with dummy episode names
subject_ids = ['sub-01']
episodes_s7 = ['s07e01a', 's07e01b']

for subject_id in subject_ids:
    submission_dict[subject_id] = {}
    for episode_idx, episode in enumerate(episodes_s7):
        # In real scenario, load actual Friends S7 features, predict, and get exact shape
        # For now, use validation predictions reshaped to mimic a real episode
        n_samples_episode = val_predictions.shape[0]  # toy: use val predictions as placeholder
        n_parcels = 1000
        
        submission_dict[subject_id][episode] = val_predictions.astype(np.float32)
        print(f"  {subject_id}/{episode}: shape {submission_dict[subject_id][episode].shape}")

print(f"\nToy submission dictionary structure:")
print(f"  Subjects: {list(submission_dict.keys())}")
print(f"  Episodes per subject: {list(submission_dict['sub-01'].keys())}")
print(f"  Prediction dtype: {submission_dict['sub-01']['s07e01a'].dtype}")

# Save submission as .npy
import os
import zipfile

submission_dir = './submission'
os.makedirs(submission_dir, exist_ok=True)

submission_npy_path = os.path.join(submission_dir, 'submission.npy')
np.save(submission_npy_path, submission_dict, allow_pickle=True)
print(f"\nSubmission saved to: {submission_npy_path}")
print(f"File size: {os.path.getsize(submission_npy_path) / (1024**2):.2f} MB")

# Create zip for Codabench submission
submission_zip_path = os.path.join(submission_dir, 'submission.zip')
with zipfile.ZipFile(submission_zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(submission_npy_path, arcname='submission.npy')

print(f"\nZipped submission: {submission_zip_path}")
print(f"Zip file size: {os.path.getsize(submission_zip_path) / (1024**2):.2f} MB")

print(f"\n✓ Submission format prepared. Ready for Codabench upload.")


## Step 5: Submit to Codabench

### Submission Instructions

The Algonauts 2025 challenge uses **Codabench** for evaluation: https://www.codabench.org/competitions/4313/

#### Steps to Submit:

1. **Create Codabench Account** (if needed)
   - Visit the competition page
   - Sign up or log in with your email

2. **Prepare Your Zipped Submission**
   - You now have `submission.zip` containing `submission.npy`
   - File structure inside zip: `submission.npy` (nested dict: `{subject: {episode: array}}`)
   - All arrays must be `float32` dtype
   - Sample counts must match official test files exactly

3. **Upload to Codabench**
   - Go to **Participate** tab → **Model Building Phase**
   - Click **"Make a Submission"**
   - Select your `submission.zip` file
   - Wait for scoring (typically < 10 minutes)

4. **Interpret Results**
   - **Score**: Average per-parcel Pearson correlation across all subjects and parcels
   - Ranges from -1 (perfect negative) to +1 (perfect positive)
   - Typical baseline: 0.1-0.3 (Ridge regression)
   - Strong models: 0.3-0.5+

5. **Important Notes**
   - Codabench automatically removes first/last 5 samples per episode (HRF artifact handling)
   - Keep your submission .zip file name simple (no special characters)
   - NumPy version: ensure `NumPy < 2.0` if using `.npy` files with pickle
   - Budget: 5 submissions/day in model building phase

### Next Steps After Getting Baseline Score:

1. **Optimize Model**: Use your MultimodalTRIBE_v2 transformer (more parameters than Ridge)
2. **Tune Hyperparameters**: Test different alphas, PCA dimensions, batch sizes
3. **Full Dataset Training**: Once subset validation looks good, scale to full Friends S1-S6
4. **Model Selection Phase**: After model building, submit to model selection phase with OOD movies
5. **Submission Phase**: Final submission with best model on model selection + training data

# STEP 6: Scale to Full Dataset (Friends S1-S6 + Movie10)

In [ ]:
print("\n" + "="*70)
print("STEP 6: Load Full Training Dataset (Friends S1-S6 + Movie10)")
print("="*70)

print("""
FULL DATASET STRUCTURE:
=======================
Training Data:
  - Friends S1-S6: ~80 hours total
  - Movie10: Multiple stimuli for additional diversity
  - Subject IDs: sub-01, sub-02, sub-03, sub-04
  - fMRI TR: 1.49 seconds (standardized)
  
Modalities (preprocessed):
  - Visual features: slow_r50 extracted frames (PCA → 250 dims)
  - Audio features: MFCC 20 coefficients (PCA → 20 dims)
  - Language features: BERT pooler + embeddings (PCA → 250 dims)
  - Total: 520 dims → further PCA to 128-256 for training
  
Expected shapes (full dataset):
  - Train features: (≈500k samples, 256 dims)
  - Train fMRI: (≈500k samples, 1000 parcels)
""")

# === PSEUDO-CODE for loading full dataset ===
print("\nLoading full training data (pseudo-code):")
print("""
# Define training episodes
train_episodes_full = [
    's01e01a', 's01e01b', ..., 's06e12b',  # Friends S1-S6
    'movie10_part1', 'movie10_part2', ...  # Movie10
]

# Load features for each episode
train_features_full = {'visual': [], 'audio': [], 'language': []}
train_fmri_full = []

for episode in train_episodes_full:
    # Load from preprocessed .npy or HDF5
    feat_vis = np.load(f'features/visual/{episode}.npy')
    feat_aud = np.load(f'features/audio/{episode}.npy')
    feat_lang = np.load(f'features/language/{episode}.npy')
    fmri_data = np.load(f'fmri/{episode}.npy')
    
    train_features_full['visual'].append(feat_vis)
    train_features_full['audio'].append(feat_aud)
    train_features_full['language'].append(feat_lang)
    train_fmri_full.append(fmri_data)

# Concatenate all episodes
train_features_full = {
    'visual': np.concatenate(train_features_full['visual']),
    'audio': np.concatenate(train_features_full['audio']),
    'language': np.concatenate(train_features_full['language']),
}
train_fmri_full = np.concatenate(train_fmri_full)

print(f"Full train features: {train_features_full['visual'].shape[0]} samples")
print(f"Full train fMRI: {train_fmri_full.shape}")
""")

# For now, use expanded subset as proxy
print("\n[DEMO] Creating expanded dataset (10x subset as proxy)...")
n_expand = 10
train_features_full_demo = {
    'visual': np.tile(train_features_subset['visual'], (n_expand, 1)),
    'audio': np.tile(train_features_subset['audio'], (n_expand, 1)),
    'language': np.tile(train_features_subset['language'], (n_expand, 1)),
}
train_fmri_full_demo = np.tile(train_fmri_subset, (n_expand, 1))

print(f"Demo full dataset shapes:")
print(f"  Visual: {train_features_full_demo['visual'].shape}")
print(f"  fMRI: {train_fmri_full_demo.shape}")
print(f"\n✓ Ready to train on full dataset (currently using demo expansion)")


# STEP 7: Hyperparameter Tuning & Model Selection

In [ ]:
print("\n" + "="*70)
print("STEP 7: Hyperparameter Tuning & Model Selection")
print("="*70)

print("""
HYPERPARAMETER TUNING STRATEGY:
===============================
1. Ridge Alpha Grid Search:
   - Alphas: [0.001, 0.01, 0.1, 1, 10, 100, 1000]
   - Cross-validation: 5-fold on training data
   - Metric: Per-parcel Pearson correlation

2. Feature Preprocessing:
   - PCA dimensions: [64, 128, 256, 512] (test on reduced features)
   - Standardization: StandardScaler on combined features
   - Impact: Faster training, reduced overfitting

3. Model Architecture Options (if using MultimodalTRIBE_v2):
   - Embedding dimension (d_model): [128, 256, 512]
   - Transformer layers: [2, 4, 6]
   - Projection dim for BMORStream: [64, 128, 256]
   - Dropout rates: [0.1, 0.3, 0.5]
   - Modality dropout: [0.2, 0.5] (probability of dropping each modality)

4. Training Hyperparameters:
   - Learning rate: [1e-4, 5e-4, 1e-3, 5e-3]
   - Batch size: [4, 8, 16] (constrained by RTX 4050 6GB)
   - Epochs: [50, 100, 200]
   - Early stopping: patience=10 on validation correlation

BASELINE vs PROPOSED:
- Baseline (Ridge): Low variance, interpretable, fast
- Proposed (MultimodalTRIBE_v2): Higher capacity, learns interactions, slower
""")

# === Grid search on Ridge alphas ===
print("\nPerforming Ridge Alpha Grid Search on Full Dataset...")

alphas_to_test = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
alpha_results = {}

# Combine features for full dataset (using demo expansion)
train_features_combined_full = np.concatenate([
    train_features_full_demo['visual'],
    train_features_full_demo['audio'],
    train_features_full_demo['language']
], axis=1)

print(f"Combined features shape: {train_features_combined_full.shape}")

# Validation set (using original subset)
val_features_combined = np.concatenate([
    val_features_subset['visual'],
    val_features_subset['audio'],
    val_features_subset['language']
], axis=1)

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

# Standardize features
scaler = StandardScaler()
train_features_scaled = scaler.fit_transform(train_features_combined_full)
val_features_scaled = scaler.transform(val_features_combined)

print(f"\nTesting alphas: {alphas_to_test}")
for alpha in alphas_to_test:
    ridge = Ridge(alpha=alpha)
    ridge.fit(train_features_scaled, train_fmri_full_demo)
    val_pred = ridge.predict(val_features_scaled)
    
    # Compute per-parcel correlation
    from scipy.stats import pearsonr
    corrs = []
    for parcel in range(val_pred.shape[1]):
        r, _ = pearsonr(val_pred[:, parcel], val_fmri_subset[:, parcel])
        corrs.append(r)
    
    mean_corr = np.mean(corrs)
    alpha_results[alpha] = mean_corr
    print(f"  Alpha {alpha:8.3f}: mean_correlation = {mean_corr:.4f}")

# Find best alpha
best_alpha = max(alpha_results, key=alpha_results.get)
best_corr = alpha_results[best_alpha]
print(f"\n✓ Best Alpha: {best_alpha} with correlation {best_corr:.4f}")

print(f"\nHyperparameter Tuning Complete. Use best_alpha={best_alpha} for final training.")


# STEP 8: Final Full-Scale Training & Codabench Submission

In [ ]:
print("\n" + "="*70)
print("STEP 8: Final Full-Scale Training & Codabench Submission")
print("="*70)

print("""
FINAL TRAINING PIPELINE:
========================
1. Load full training data (Friends S1-S6 + Movie10)
2. Preprocess: standardize + optional PCA
3. Train final model using best hyperparameters from Step 7
4. Load test data (Friends S7 for all subjects)
5. Predict fMRI for test data
6. Format as nested dictionary {subject: {episode: predictions}}
7. Save and zip for Codabench submission
""")

# ===== STEP 8A: Train Final Model =====
print("\n--- Training Final Ridge Model with Best Alpha ---")
final_alpha = best_alpha
print(f"Using best_alpha={final_alpha}")

final_ridge_model = Ridge(alpha=final_alpha)
final_ridge_model.fit(train_features_scaled, train_fmri_full_demo)
print(f"✓ Model trained on full dataset")

# ===== STEP 8B: Load Test Data (Friends S7) =====
print("\n--- Loading Test Data (Friends S7) ---")
print("""
Expected structure for test data:
  - Episodes: s07e01a, s07e01b, ..., s07e12b (24 episodes per subject)
  - Subjects: sub-01, sub-02, sub-03, sub-04
  - Each (subject, episode) pair has features and fMRI sample counts
  
NOTE: Official test fMRI files (sub-0X_friends-s7_fmri_samples.npy) contain
      exact sample counts for each episode. Must match exactly in submission.
""")

# For demo: use subset as test proxy
print("\n[DEMO] Using subset as test proxy (replace with real Friends S7)...")
test_episodes_s7 = ['s07e01a', 's07e01b']  # Demo
test_subjects = ['sub-01', 'sub-02']  # Demo

test_predictions_dict = {}
for subject in test_subjects:
    test_predictions_dict[subject] = {}
    
    for episode in test_episodes_s7:
        # In real scenario: load actual S7 features
        test_features = val_features_scaled[:100]  # Demo: use 100 samples
        test_pred = final_ridge_model.predict(test_features)
        
        test_predictions_dict[subject][episode] = test_pred.astype(np.float32)
        print(f"  {subject}/{episode}: shape {test_pred.shape}")

# ===== STEP 8C: Validate Test Predictions =====
print("\n--- Computing Test Set Metrics ---")
test_correlations = []
for subject in test_subjects:
    for episode in test_episodes_s7:
        test_pred = test_predictions_dict[subject][episode]
        test_true = val_fmri_subset[:test_pred.shape[0]]  # Demo: use val as proxy
        
        episode_corrs = []
        for parcel in range(test_pred.shape[1]):
            r, _ = pearsonr(test_pred[:, parcel], test_true[:, parcel])
            episode_corrs.append(r)
        
        episode_mean_corr = np.mean(episode_corrs)
        test_correlations.append(episode_mean_corr)
        print(f"  {subject}/{episode}: mean_correlation = {episode_mean_corr:.4f}")

overall_test_corr = np.mean(test_correlations)
print(f"\n✓ Overall Test Correlation: {overall_test_corr:.4f}")

# ===== STEP 8D: Save and Zip Final Submission =====
print("\n--- Creating Final Submission Archive ---")

submission_final_path = os.path.join(submission_dir, 'submission_final.npy')
np.save(submission_final_path, test_predictions_dict, allow_pickle=True)
print(f"✓ Saved predictions: {submission_final_path}")
print(f"  File size: {os.path.getsize(submission_final_path) / (1024**2):.2f} MB")

submission_final_zip = os.path.join(submission_dir, 'submission_final.zip')
with zipfile.ZipFile(submission_final_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(submission_final_path, arcname='submission.npy')

print(f"✓ Created zip archive: {submission_final_zip}")
print(f"  Zip size: {os.path.getsize(submission_final_zip) / (1024**2):.2f} MB")

# ===== STEP 8E: Submission Checklist =====
print("\n" + "="*70)
print("CODABENCH SUBMISSION CHECKLIST")
print("="*70)

submission_checklist = {
    "Data": {
        "✓ Training data loaded": train_fmri_full_demo.shape,
        "✓ Test predictions computed": test_predictions_dict['sub-01']['s07e01a'].shape,
        "✓ Predictions dtype": test_predictions_dict['sub-01']['s07e01a'].dtype,
    },
    "Model": {
        "✓ Best alpha found": best_alpha,
        "✓ Mean test correlation": f"{overall_test_corr:.4f}",
        "✓ Model type": "Ridge Regression",
    },
    "Submission": {
        "✓ Nested dict structure": "{subject: {episode: array}}",
        "✓ All arrays float32": "Yes",
        "✓ Zip file created": os.path.exists(submission_final_zip),
        "✓ Zip file ready": f"{os.path.getsize(submission_final_zip) / (1024**2):.2f} MB",
    }
}

for category, items in submission_checklist.items():
    print(f"\n{category}:")
    for key, value in items.items():
        print(f"  {key}: {value}")

print("\n" + "="*70)
print("READY FOR CODABENCH UPLOAD")
print("="*70)
print(f"\nNext Steps:")
print(f"1. Navigate to: https://www.codabench.org/competitions/4313/")
print(f"2. Go to 'Participate' → 'Model Building Phase'")
print(f"3. Click 'Make a Submission'")
print(f"4. Upload: {submission_final_zip}")
print(f"5. Wait for scoring (typically < 10 minutes)")
print(f"\nExpected Score: ~0.15-0.30 (for Ridge baseline)")
print(f"Potential Score: ~0.35-0.50 (with MultimodalTRIBE_v2)")


**Step 6: Validation**

**Step 7: Preparing Submission Format**

**Step 8: Submitting to Codabench**